In [ ]:
!pip install "flwr<1.13.0" ray

from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')

  Using cached protobuf-4.25.8-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.8-cp37-abi3-manylinux2014_x86_64.whl (294 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.28.3
    Uninstalling protobuf-5.28.3:
      Successfully uninstalled protobuf-5.28.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-health-checking 1.78.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
tensorflow-text 2.19.0 requires tensorflow<2.20,>=2.19.0, but you have tensorflow 2.18.0 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.18.0 which is incompatible.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19,

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install "jax<=0.4.33" "jaxlib<=0.4.33" tensorflow==2.18.0 protobuf==5.28.3

In [ ]:
import numpy as np
import zipfile
import io
from pathlib import Path
import matplotlib.pyplot as plt

# ── Point this at ONE of your Town zip files ──────────────────────────
#CHANNEL_ZIP = Path("/Users/sadmanrahin/Library/CloudStorage/GoogleDrive-saddyrahin2004@gmail.com/My Drive/Dataset/sunny/Channel Data/V2I/Nt_1_16_Nr_1_16_fc_28GHz/Town03.zip")
CHANNEL_ZIP = Path("/content/drive/MyDrive/Dataset/sunny/Channel Data/V2I/Nt_1_16_Nr_1_16_fc_28GHz/Town03.zip")

print("=" * 70)
print("CHANNEL ZIP CONTENTS")
print("=" * 70)

with zipfile.ZipFile(CHANNEL_ZIP, "r") as z:
    all_files = z.namelist()
    npz_files = [f for f in all_files if f.endswith("_paths.npz")]
    yaml_files = [f for f in all_files if f.endswith(".yaml")]
    print(f"  .npz files  : {len(npz_files)}")
    print(f"  .yaml files : {len(yaml_files)}")
    print(f"  First 5 npz : {npz_files[:5]}")

    # Load the first .npz for inspection
    raw = z.read(npz_files[0])

npz = np.load(io.BytesIO(raw))
print("\n" + "=" * 70)
print(f"NPZ FILE: {npz_files[0]}")
print("=" * 70)
print(f"  Keys: {npz.files}")
for k in npz.files:
    arr = npz[k]
    print(f"  [{k}]  shape={arr.shape}  dtype={arr.dtype}  "
          f"min={np.abs(arr).min():.4g}  max={np.abs(arr).max():.4g}")


CHANNEL ZIP CONTENTS
  .npz files  : 15700
  .yaml files : 15700
  First 5 npz : ['Town03/Town03_Tjunction/cav_1/004276_paths.npz', 'Town03/Town03_Tjunction/cav_1/004283_paths.npz', 'Town03/Town03_Tjunction/cav_1/004284_paths.npz', 'Town03/Town03_Tjunction/cav_1/004288_paths.npz', 'Town03/Town03_Tjunction/cav_1/004291_paths.npz']

NPZ FILE: Town03/Town03_Tjunction/cav_1/004276_paths.npz
  Keys: ['a', 'tau', 'theta_t', 'phi_t', 'theta_r', 'phi_r', 'glob_phi_t', 'glob_phi_r', 'glob_theta_t', 'glob_theta_r']
  [a]  shape=(1, 1, 16, 1, 16, 3, 1)  dtype=complex64  min=1.932e-06  max=3.076e-05
  [tau]  shape=(1, 1, 1, 3)  dtype=float32  min=0  max=8.499e-07
  [theta_t]  shape=(1, 1, 1, 3)  dtype=float32  min=1.591  max=1.726
  [phi_t]  shape=(1, 1, 1, 3)  dtype=float32  min=1.215  max=2.458
  [theta_r]  shape=(1, 1, 1, 3)  dtype=float32  min=1.192  max=1.792
  [phi_r]  shape=(1, 1, 1, 3)  dtype=float32  min=0.4003  max=3.115
  [glob_phi_t]  shape=(1, 1, 1, 3)  dtype=float32  min=0.6834  max=

## 1. Imports & Reproducibility

In [ ]:
import numpy as np
import random
import os
import re
import io
import zipfile
import pickle
import hashlib
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path, PurePosixPath
from typing import Dict, List, Optional, Sequence, Tuple, Union

import flwr as fl
import ray
import tensorflow as tf
from tensorflow import keras
import pandas as pd

# ── Reproducibility ───────────────────────────────────────────────────
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["PYTHONHASHSEED"] = str(seed)
tf.keras.backend.clear_session()

results_dir = "experiment_results"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print("Random seeds set. TF version:", tf.__version__)

Random seeds set. TF version: 2.18.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 2. Configuration

In [ ]:
CFG = {
    # ── Training ──────────────────────────────────────────────────────────────
    "local_epochs"   : 3,
    "lr"             : 1e-3,
    "batch_size"     : 64,
    "grad_clip_norm" : 5.0,
    "label_smoothing": 0.15,

    # ── Federated ─────────────────────────────────────────────────────────────
    "client_frac"    : 0.9,
    "fedprox_mu"     : 0.01,   # FedProx proximal term (A2G non-IID drift control)

    # ── Codebook ──────────────────────────────────────────────────────────────
    # UAV: A2G channels are elevation-dominant. Current horizontal DFT codebook
    # (Nt=16) gives Q_tx=16. For 3D UAV codebook: Q_tx = 8_az × 8_el = 64.
    "Q_tx"           : 16,
    "Q_rx"           : 16,

    # ── Loss head weights ─────────────────────────────────────────────────────
    "snr_loss_weight"   : 0.5,   # UAV SNR head (5th task)
    # NOTE: pos_weight_los REMOVED — LoS head eliminated (84% NLOS imbalance;
    # head was redundant with gopt signal: LOS <-> high beamforming gain).
    "pos_weight_change" : 5.0,   # beam_change is ~equally balanced

    # ── DDIL (Degraded/Disconnected/Intermittent/Limited) environment ─────────
    "ddil_packet_dropout"      : 0.10,  # 10% uplink packet loss per frame
    "ddil_doppler_noise_std"   : 0.05,  # A2G Doppler at 80 km/h @ 28 GHz
    "ddil_signal_blockage_prob": 0.05,  # abrupt NLOS transition probability
    "ddil_connectivity_prob"   : 0.90,  # P(UAV connects to server each round)
}

MAX_CLIENTS = None
print(f"CFG: {CFG}")
print(f"MAX_CLIENTS: {MAX_CLIENTS}  (None = use all)")
print(f"\nDDIL env: pkt_drop={CFG['ddil_packet_dropout']*100:.0f}%  "
      f"doppler_std={CFG['ddil_doppler_noise_std']}  "
      f"blockage={CFG['ddil_signal_blockage_prob']*100:.0f}%  "
      f"connectivity={CFG['ddil_connectivity_prob']*100:.0f}%")

## 3. Dataset

In [ ]:
def _parse_total_antennas(config_name: str, side: str) -> int:
    m = re.search(rf"{side}_(\d+)_(\d+)", config_name)
    if not m:
        raise ValueError(f"Could not parse {side} from: {config_name}")
    return int(m.group(1)) * int(m.group(2))


def _safe_isfinite(x: np.ndarray) -> bool:
    if np.iscomplexobj(x):
        return np.isfinite(x.real).all() and np.isfinite(x.imag).all()
    return np.isfinite(x).all()


@dataclass(frozen=True)
class ChannelSampleRef:
    zip_path: Path
    inner_npz: str


def generate_dft_codebook(size: int, num_beams: int) -> np.ndarray:
    """
    DFT steering-vector codebook  (MMW paper Eq. 2 / Eq. 4).

    f(q)[n] = (1 / sqrt(Nt)) * exp(j * 2*pi / Q * n * q)

    Parameters
    ----------
    size      : Nt  – number of Tx antennas
    num_beams : Q   – number of DFT beams (paper uses Q=64)

    Returns
    -------
    codebook : complex64 array of shape (size, num_beams)
               Column q is the steering vector for beam q.
    """
    n = np.arange(size).reshape(-1, 1)
    q = np.arange(num_beams).reshape(1, -1)
    codebook = (1.0 / np.sqrt(size)) * np.exp(
        1j * (2 * np.pi / num_beams) * n * q
    )
    return codebook.astype(np.complex64)


# Sensor frame constants
# LiDAR reduced 512→128 pts: 4× lower RAM; sufficient for A2G spatial coverage.
_N_LIDAR_PTS = 128
_IMU_DIM     = 7   # acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, compass

print("Dataset helpers defined.")
print(f"Sensor constants: _N_LIDAR_PTS={_N_LIDAR_PTS}, _IMU_DIM={_IMU_DIM}")

Dataset helpers defined.
Sensor constants: _N_LIDAR_PTS=128, _IMU_DIM=7


In [ ]:
# Cell: ChannelDataset class (multi-weather, multi-town, all CAVs)
class ChannelDataset:
    """
    Channel-only dataset aligned with MMW paper methodology.
    Supports loading from MULTIPLE weather conditions and towns.

    Feature: Effective channel H = sum_paths(a), shape (Nr, Nt)
             stacked as [real(H), imag(H), |H|]  -> (Nr, Nt, 3)

    Labels (multi-task):
        beam_index : int   — argmax exhaustive beam search (paper Eq. 4)
        g_opt      : float — peak achievable beamforming gain (channel quality)
        los        : int   — 1 = LoS, 0 = NLoS (from NPZ or power-ratio heuristic)
        H_complex  : complex64 (Nr, Nt)
    """

    # Known path variants for different weather conditions
    CHANNEL_PATH_VARIANTS = [
        "Channel Data/V2I",   # sunny layout
        "Channel Data",       # foggy layout (no V2I subdirectory)
    ]

    def __init__(
        self,
        root: Union[str, Path],
        config_name: str,
        Q_tx: int = 16,
        Q_rx: int = 16,
        weather_conditions: Optional[Sequence[str]] = None,
        towns: Optional[Sequence[str]] = None,
        scenario_contains: Optional[str] = None,
        cav_contains: Optional[str] = None,
        stride: int = 1,
        assert_no_nans: bool = True,
        assert_shapes: bool = True,
    ):
        self.root = Path(root)
        self.config_name = config_name

        if weather_conditions is None:
            weather_conditions = ["sunny"]
        self.weather_conditions = list(weather_conditions)

        self.config_dirs = []
        for weather in self.weather_conditions:
            found = False
            for variant in self.CHANNEL_PATH_VARIANTS:
                cdir = self.root / weather / variant / config_name
                if cdir.exists():
                    self.config_dirs.append(cdir)
                    print(f"  [OK] Found channel data: {weather} → {cdir}")
                    found = True
                    break
            if not found:
                print(f"  [WARN] No channel data for '{weather}' — skipped")
        if not self.config_dirs:
            raise FileNotFoundError(
                f"No channel data found for any weather condition: {self.weather_conditions}"
            )

        self.nt = _parse_total_antennas(config_name, "Nt")
        self.nr = _parse_total_antennas(config_name, "Nr")
        self.Q_tx = Q_tx
        self.Q_rx = Q_rx

        self.towns = list(towns) if towns else None
        self.scenario_contains = scenario_contains
        self.cav_contains = cav_contains
        self.stride = max(1, int(stride))
        self.assert_no_nans = assert_no_nans
        self.assert_shapes = assert_shapes

        # DFT codebooks (paper Eq. 2)
        self.tx_codebook = generate_dft_codebook(self.nt, Q_tx)  # (Nt, Q_tx)
        self.rx_codebook = generate_dft_codebook(self.nr, Q_rx)  # (Nr, Q_rx)

        self.index: List[ChannelSampleRef] = self._build_index()
        if self.stride > 1:
            self.index = self.index[::self.stride]

        self._expected_csi_shape: Optional[Tuple[int, ...]] = None

    # -- Index building -------------------------------------------------------
    def _build_index(self) -> List[ChannelSampleRef]:
        refs: List[ChannelSampleRef] = []

        for config_dir in self.config_dirs:
            if self.towns is None:
                zips = sorted(config_dir.glob("Town*.zip"))
                flat_towns = sorted([p.name for p in config_dir.glob("Town*") if p.is_dir()])
            else:
                zips = [config_dir / f"{t}.zip" for t in self.towns]
                flat_towns = list(self.towns)

            seen_towns = set()

            for zp in zips:
                town = zp.stem
                seen_towns.add(town)
                flat_dir = zp.with_suffix("")   # Town03/ next to Town03.zip

                if flat_dir.exists() and any(flat_dir.rglob("*_paths.npz")):
                    # Fast path: scan flat directory — no ZIP open needed
                    for p_flat in sorted(flat_dir.rglob("*_paths.npz")):
                        name = str(PurePosixPath(p_flat.relative_to(flat_dir)))
                        if self.scenario_contains and self.scenario_contains.lower() not in name.lower():
                            continue
                        if self.cav_contains and self.cav_contains.lower() not in name.lower():
                            continue
                        refs.append(ChannelSampleRef(zip_path=zp, inner_npz=name))
                elif zp.exists():
                    # Fallback: list ZIP contents (Drive FUSE — slow)
                    with zipfile.ZipFile(zp, "r") as z:
                        for name in z.namelist():
                            if not name.endswith("_paths.npz"):
                                continue
                            p = PurePosixPath(name)
                            if self.scenario_contains and self.scenario_contains.lower() not in str(p).lower():
                                continue
                            if self.cav_contains and self.cav_contains.lower() not in str(p).lower():
                                continue
                            refs.append(ChannelSampleRef(zip_path=zp, inner_npz=name))

            # Flat-only fallback: no ZIP present, but TownXX/ exists with extracted npz files.
            for town in flat_towns:
                if town in seen_towns:
                    continue
                flat_dir = config_dir / town
                if not flat_dir.exists() or not any(flat_dir.rglob("*_paths.npz")):
                    continue
                zp = config_dir / f"{town}.zip"  # synthetic path for metadata + flat resolution
                for p_flat in sorted(flat_dir.rglob("*_paths.npz")):
                    name = str(PurePosixPath(p_flat.relative_to(flat_dir)))
                    if self.scenario_contains and self.scenario_contains.lower() not in name.lower():
                        continue
                    if self.cav_contains and self.cav_contains.lower() not in name.lower():
                        continue
                    refs.append(ChannelSampleRef(zip_path=zp, inner_npz=name))

        def sort_key(ref: ChannelSampleRef):
            p = PurePosixPath(ref.inner_npz)
            m = re.match(r"(\d+)_paths$", p.stem)
            return (str(ref.zip_path), str(p.parent), int(m.group(1)) if m else -1)

        refs.sort(key=sort_key)
        if not refs:
            raise ValueError(
                f"No *_paths.npz found under config dirs: {self.config_dirs}"
            )
        return refs

    # -- Metadata --------------------------------------------------------------
    def _parse_metadata(self, inner_path: str) -> Dict[str, str]:
        p = PurePosixPath(inner_path)
        m = re.match(r"(\d+)_paths\.npz$", p.name)
        frame_id = int(m.group(1)) if m else -1
        cav_id = p.parent.name if "cav" in p.parent.name.lower() else "unknown"
        location = p.parent.parent.name or "unknown"
        return {"location": location, "cav_id": cav_id, "frame_id": frame_id}

    def get_sample_metadata(self, idx: int) -> Dict[str, str]:
        ref = self.index[idx]
        meta = self._parse_metadata(ref.inner_npz)
        meta["town"] = ref.zip_path.stem
        meta["zip_path"] = str(ref.zip_path)
        meta["inner_path"] = ref.inner_npz
        for weather in self.weather_conditions:
            if weather in str(ref.zip_path):
                meta["weather"] = weather
                break
        else:
            meta["weather"] = "unknown"
        return meta

    def build_metadata_index(self) -> pd.DataFrame:
        rows = []
        for idx in range(len(self)):
            meta = self.get_sample_metadata(idx)
            meta["sample_idx"] = idx
            rows.append(meta)
        return pd.DataFrame(rows)

    # -- Core loading ----------------------------------------------------------
    def __len__(self) -> int:
        return len(self.index)

    def _load_npz(self, ref: ChannelSampleRef) -> Dict[str, np.ndarray]:
        # Priority 1: SSD cache (fastest — local disk, no FUSE latency)
        if getattr(self, "ssd_root", None):
            try:
                rel = ref.zip_path.with_suffix("").relative_to(self.root)
                ssd_path = Path(self.ssd_root) / rel / ref.inner_npz
                if ssd_path.exists():
                    npz = np.load(ssd_path)
                    return {k: npz[k] for k in npz.files}
            except (ValueError, Exception):
                pass
        # Priority 2: flat .npz next to ZIP on Drive (no ZIP overhead)
        flat_path = ref.zip_path.with_suffix("") / ref.inner_npz
        if flat_path.exists():
            try:
                npz = np.load(flat_path)
                return {k: npz[k] for k in npz.files}
            except (EOFError, Exception):
                pass  # corrupted flat file — fall through to ZIP
        # Priority 3: read from ZIP (original fallback)
        try:
            with zipfile.ZipFile(ref.zip_path, "r") as z:
                raw = z.read(ref.inner_npz)
            npz = np.load(io.BytesIO(raw))
            return {k: npz[k] for k in npz.files}
        except (EOFError, Exception) as e:
            raise EOFError(f"Corrupted npz: {ref.zip_path}/{ref.inner_npz}") from e

    def _extract_csi(self, arrays: Dict[str, np.ndarray]) -> np.ndarray:
        """
        Effective channel matrix as model input.
        Returns (Nr, Nt, 3) float32: [real(H), imag(H), |H|]
        """
        a = arrays["a"]
        if self.assert_no_nans:
            assert _safe_isfinite(a), "Non-finite values in 'a'"

        a_sq = np.squeeze(a).astype(np.complex64)
        if a_sq.ndim == 2:
            a_sq = a_sq[:, :, np.newaxis]
        if a_sq.ndim != 3:
            raise ValueError(f"Unexpected squeezed 'a' shape: {a_sq.shape}")

        H = np.sum(a_sq, axis=2)  # (Nr, Nt) complex64
        csi = np.stack([H.real, H.imag, np.abs(H)], axis=-1).astype(np.float32)

        if self.assert_no_nans:
            assert _safe_isfinite(csi), "Non-finite in csi"
        return csi

    def compute_beam_index(self, a_sq: np.ndarray) -> Tuple[int, float, np.ndarray]:
        """
        Exhaustive beam search (paper Eq. 4).
        Returns (beam_idx, g_opt, H) where g_opt = peak achievable gain.
        """
        H = np.sum(a_sq.astype(np.complex64), axis=2)  # (Nr, Nt)
        response = self.rx_codebook.conj().T @ H @ self.tx_codebook  # (Q_rx, Q_tx)
        gain_per_tx = np.max(np.abs(response) ** 2, axis=0)  # (Q_tx,)
        g_opt = float(gain_per_tx.max())
        return int(np.argmax(gain_per_tx)), g_opt, H

    def __getitem__(self, idx: int) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        for attempt in range(len(self.index)):
            _idx = (idx + attempt) % len(self.index)
            ref = self.index[_idx]
            try:
                arrays = self._load_npz(ref)
            except EOFError:
                if attempt == 0:
                    import warnings
                    warnings.warn(f"[ChannelDataset] Skipping corrupted file: {ref.inner_npz}")
                continue

            a_sq = np.squeeze(arrays["a"]).astype(np.complex64)
            if a_sq.ndim == 2:
                a_sq = a_sq[:, :, np.newaxis]

            csi_tensor = self._extract_csi(arrays)

            if self.assert_shapes:
                if self._expected_csi_shape is None:
                    self._expected_csi_shape = tuple(csi_tensor.shape)
                else:
                    assert tuple(csi_tensor.shape) == self._expected_csi_shape, (
                        f"CSI shape mismatch: {self._expected_csi_shape} vs {csi_tensor.shape}"
                    )

            beam_idx, g_opt, H = self.compute_beam_index(a_sq)

            # LoS: read from NPZ if present; else estimate from dominant-path power ratio
            _los_raw = arrays.get("los", None)
            if _los_raw is not None:
                los = int(np.squeeze(_los_raw).flat[0])
            else:
                path_power  = np.sum(np.abs(a_sq) ** 2, axis=(0, 1))
                total_power = path_power.sum() + 1e-12
                los = 1 if (path_power.max() / total_power) > 0.85 else 0

            labels = {
                "beam_index": np.array(beam_idx, dtype=np.int64),
                "g_opt"     : np.array(g_opt,     dtype=np.float32),
                "los"       : np.array(los,        dtype=np.int32),
                "H_complex" : H,
            }

            return csi_tensor, labels

        raise RuntimeError(f"No valid samples found in dataset (all {len(self.index)} files corrupted?)")


In [ ]:
# ── SensorIndex + load_sensor_frame + MultimodalChannelDataset ────────────────
# Modalities: CSI + LiDAR-512 + IMU  (RGB and GPS dropped)
import yaml
import struct

# ---------------------------------------------------------------------------
#  SensorIndex — discovers sensor zips and resolves channel→sensor alignment
# ---------------------------------------------------------------------------

SENSOR_PATH_VARIANTS = [
    "Sensor Data",
]

@dataclass
class SensorZipEntry:
    """One sensor .zip  (e.g. Town03_Tjunction_wiz_slope_seed42.zip)."""
    zip_path: Path
    scenario_prefix: str        # e.g. "Town03_Tjunction_wiz_slope_seed42"
    channel_location: str       # e.g. "Town03_Tjunction"  (prefix match)
    town: str                   # e.g. "Town03"
    is_zip: bool = True


class SensorIndex:
    """
    Maps  (weather, channel_inner_npz)  →  (SensorZipEntry, cav_id, frame_id)
    so that MultimodalChannelDataset can look up the sensor zip for every
    channel sample.
    """

    def __init__(self, root: Union[str, Path],
                 weather_conditions: Sequence[str],
                 towns: Sequence[str]):
        self.root = Path(root)
        # key = (weather, channel_location, town)  →  SensorZipEntry
        self._map: Dict[Tuple[str, str, str], SensorZipEntry] = {}
        self._build(weather_conditions, towns)

    # ── build ----------------------------------------------------------------
    def _build(self, weathers, towns):
        for weather in weathers:
            for variant in SENSOR_PATH_VARIANTS:
                for town in towns:
                    sensor_dir = self.root / weather / variant / town
                    if not sensor_dir.exists():
                        continue
                    # 2) Flat-extracted scenario directories
                    for d in sorted([p for p in sensor_dir.iterdir() if p.is_dir()]):
                        stem = d.name
                        ch_loc = self._stem_to_channel_location(stem, town)
                        entry = SensorZipEntry(
                            zip_path=d,
                            scenario_prefix=stem,
                            channel_location=ch_loc,
                            town=town,
                            is_zip=False,
                        )
                        key = (weather, ch_loc, town)
                        # Do not overwrite an existing ZIP entry; prefer ZIP when both exist.
                        self._map[key] = entry   # flat always wins
                    # 2) ZIP archives
                    for zp in sorted(sensor_dir.glob("*.zip")):
                        stem = zp.stem  # e.g. Town03_Tjunction_wiz_slope_seed42
                        # derive the channel_location by stripping everything
                        # after the town-specific location tag.
                        # Channel uses e.g. "Town03_Tjunction"
                        # Sensor zip is    "Town03_Tjunction_wiz_slope_seed42"
                        ch_loc = self._stem_to_channel_location(stem, town)
                        entry = SensorZipEntry(
                            zip_path=zp,
                            scenario_prefix=stem,
                            channel_location=ch_loc,
                            town=town,
                            is_zip=True,
                        )
                        key = (weather, ch_loc, town)
                        self._map.setdefault(key, entry)

        n = len(self._map)
        n_zip = sum(1 for v in self._map.values() if v.is_zip)
        n_flat = n - n_zip
        print(f"[SensorIndex] {n} sensor sources indexed ({n_zip} zip + {n_flat} flat)")

    @staticmethod
    def _stem_to_channel_location(stem: str, town: str) -> str:
        """
        Town03_Tjunction_wiz_slope_seed42  →  Town03_Tjunction
        Town03_5wayroad_seed28             →  Town03_5wayroad
        Heuristic: the channel location = town + '_' + next word.
        """
        # Remove the town prefix to isolate the location part
        rest = stem[len(town) + 1:]  # "Tjunction_wiz_slope_seed42"
        # Take everything up to the first _seed or _wiz or similar suffix
        # Strategy: split on '_' and take parts until we hit a known suffix
        parts = rest.split("_")
        loc_parts = []
        stop_words = {"wiz", "seed", "slope"}
        for p in parts:
            if p.lower() in stop_words or re.match(r"seed\d+", p.lower()):
                break
            loc_parts.append(p)
        location = "_".join(loc_parts)  # "Tjunction" or "5wayroad" etc.
        return f"{town}_{location}"

    # ── resolve ---------------------------------------------------------------
    def resolve(self, weather: str, channel_inner_npz: str):
        """
        Given a channel inner path like
            Town03/Town03_Tjunction/cav_1/004276_paths.npz
        return (SensorZipEntry, cav_id, frame_id) or None.
        """
        p = PurePosixPath(channel_inner_npz)
        # parts: ('Town03', 'Town03_Tjunction', 'cav_1', '004276_paths.npz')
        if len(p.parts) < 4:
            return None
        town = p.parts[0]
        ch_location = p.parts[1]
        cav_id = p.parts[2]
        m = re.match(r"(\d+)_paths\.npz$", p.parts[3])
        if not m:
            return None
        frame_id = m.group(1)   # string like "004276"

        key = (weather, ch_location, town)
        entry = self._map.get(key)
        if entry is None:
            return None
        return (entry, cav_id, frame_id)


# ---------------------------------------------------------------------------
#  load_sensor_frame — reads LiDAR + IMU from a sensor zip
# ---------------------------------------------------------------------------

def _read_pcd_xyz(raw_bytes: bytes) -> np.ndarray:
    """Parse ASCII PCD → (N, 3) float32 (only xyz, ignore rgb)."""
    lines = raw_bytes.split(b"\n")
    header_end = 0
    n_points = 0
    for i, line in enumerate(lines):
        text = line.decode(errors="replace").strip()
        if text.startswith("POINTS"):
            n_points = int(text.split()[1])
        if text == "DATA ascii":
            header_end = i + 1
            break
    pts = []
    for line in lines[header_end: header_end + n_points]:
        parts = line.split()
        if len(parts) >= 3:
            pts.append([float(parts[0]), float(parts[1]), float(parts[2])])
    return np.array(pts, dtype=np.float32) if pts else np.zeros((0, 3), np.float32)


def load_sensor_frame(
    entry: SensorZipEntry,
    cav_id: str,
    frame_id: str,
    n_lidar_pts: int = _N_LIDAR_PTS,
    rng: np.random.Generator = None,
) -> Dict[str, np.ndarray]:
    """
    Load LiDAR and IMU from sensor data source (ZIP or flat folder).

    Sensor zip layout:
        <scenario>/<cav_id>/<frame_id>.pcd   → LiDAR point cloud
        <scenario>/<cav_id>/<frame_id>.yaml  → IMU (imu_measurement section)

    Returns
    -------
    {"lidar": (n_lidar_pts, 3) float32, "imu": (7,) float32}
    IMU layout: [acc_x/g, acc_y/g, acc_z/g, gyro_x, gyro_y, gyro_z, compass/360]
    """
    if rng is None:
        rng = np.random.default_rng(42)

    prefix = f"{entry.scenario_prefix}/{cav_id}/{frame_id}"

    if entry.is_zip:
        with zipfile.ZipFile(entry.zip_path, "r") as z:
            pcd_raw = z.read(f"{prefix}.pcd")
            yaml_raw = z.read(f"{prefix}.yaml")
    else:
        # Flat layout can be either:
        #   <scenario>/<cav>/<frame>.pcd  or  <scenario>/<scenario>/<cav>/<frame>.pcd
        p = Path(entry.zip_path)
        cand1 = p / cav_id / f"{frame_id}.pcd"
        cand2 = p / entry.scenario_prefix / cav_id / f"{frame_id}.pcd"
        pcd_path = cand1 if cand1.exists() else cand2

        y1 = p / cav_id / f"{frame_id}.yaml"
        y2 = p / entry.scenario_prefix / cav_id / f"{frame_id}.yaml"
        yaml_path = y1 if y1.exists() else y2

        pcd_raw = pcd_path.read_bytes()
        yaml_raw = yaml_path.read_bytes()

    # ── LiDAR ─────────────────────────────────────────────────────────────
    pts = _read_pcd_xyz(pcd_raw)
    if len(pts) == 0:
        pts = np.zeros((1, 3), np.float32)
    if len(pts) >= n_lidar_pts:
        idx = rng.choice(len(pts), n_lidar_pts, replace=False)
        pts = pts[idx]
    else:
        pad = np.zeros((n_lidar_pts - len(pts), 3), np.float32)
        pts = np.concatenate([pts, pad], axis=0)
    mean = pts.mean(axis=0, keepdims=True)
    std  = pts.std(axis=0, keepdims=True) + 1e-8
    lidar = ((pts - mean) / std).astype(np.float32)

    # ── IMU (from YAML) ───────────────────────────────────────────────────
    meta = yaml.safe_load(yaml_raw)
    imu_meta = meta["sensors"]["imu_measurement"]
    if imu_meta.get("data_missing", False):
        imu = np.zeros(_IMU_DIM, np.float32)
    else:
        acc  = imu_meta["accelerometer"]
        gyro = imu_meta["gyroscope"]
        comp = float(imu_meta.get("compass", 0.0))
        imu  = np.array([
            acc["x"]  / 9.81,  acc["y"]  / 9.81,  acc["z"]  / 9.81,
            gyro["x"],         gyro["y"],          gyro["z"],
            comp / 360.0,
        ], dtype=np.float32)

    return {"lidar": lidar, "imu": imu}


# ---------------------------------------------------------------------------
#  MultimodalChannelDataset
# ---------------------------------------------------------------------------
class MultimodalChannelDataset:
    """
    Aligns channel samples with their sensor observations.

    __getitem__ returns:
        csi    : (Nr, Nt, 3)          float32
        lidar  : (_N_LIDAR_PTS, 3)    float32  normalised xyz  (512 pts)
        imu    : (_IMU_DIM,)          float32  [acc/g ×3, gyro ×3, compass/360]
        labels : {"beam_index": int64, "H_complex": complex64 (Nr,Nt)}
    """

    def __init__(
        self,
        channel_dataset: "ChannelDataset",
        sensor_index: SensorIndex,
        n_lidar_pts: int = _N_LIDAR_PTS,
        seed: int = 42,
    ):
        self.channel_ds  = channel_dataset
        self.sensor_idx  = sensor_index
        self.n_lidar_pts = n_lidar_pts
        self._rng        = np.random.default_rng(seed)

        self._sensor_map = self._build_sensor_map()
        matched = sum(1 for v in self._sensor_map if v is not None)
        print(f"[MultimodalDataset] {len(self)} samples  |"
              f"  sensor-aligned: {matched}  |"
              f"  fallback (zeros): {len(self) - matched}")

    def _build_sensor_map(self):
        mapping = []
        for idx in range(len(self.channel_ds)):
            ref  = self.channel_ds.index[idx]
            meta = self.channel_ds.get_sample_metadata(idx)
            weather = meta.get("weather", "sunny")
            resolved = self.sensor_idx.resolve(weather, ref.inner_npz)
            mapping.append(resolved)
        return mapping

    def __len__(self):
        return len(self.channel_ds)

    def __getitem__(self, idx: int):
        csi, labels = self.channel_ds[idx]

        resolved = self._sensor_map[idx]
        if resolved is not None:
            entry, cav_id, frame_id = resolved
            try:
                sensor = load_sensor_frame(
                    entry, cav_id, frame_id,
                    n_lidar_pts=self.n_lidar_pts,
                    rng=self._rng,
                )
                lidar = sensor["lidar"]
                imu   = sensor["imu"]
            except Exception:
                lidar, imu = self._zeros()
        else:
            lidar, imu = self._zeros()

        return csi, lidar, imu, labels

    def _zeros(self):
        lidar = np.zeros((self.n_lidar_pts, 3), np.float32)
        imu   = np.zeros(_IMU_DIM, np.float32)
        return lidar, imu

    @property
    def tx_codebook(self): return self.channel_ds.tx_codebook
    @property
    def rx_codebook(self): return self.channel_ds.rx_codebook
    @property
    def nr(self): return self.channel_ds.nr
    @property
    def nt(self): return self.channel_ds.nt


print("SensorIndex, load_sensor_frame, MultimodalChannelDataset defined.")

SensorIndex, load_sensor_frame, MultimodalChannelDataset defined.


In [ ]:
class DatasetSplitter:
    """Trajectory-aware train/test splitting (1 client = 1 CAV trajectory)."""

    def __init__(self, dataset: ChannelDataset):
        self.dataset = dataset
        self.metadata_df = dataset.build_metadata_index()
        print(f"Metadata index built: {len(self.metadata_df)} samples")
        # Summary stats
        n_towns = self.metadata_df["town"].nunique()
        n_weather = self.metadata_df["weather"].nunique() if "weather" in self.metadata_df.columns else 1
        print(f"  Towns: {n_towns}, Weather conditions: {n_weather}")

    def get_trajectory_groups(self) -> Dict[str, List[int]]:
        trajectories = {}
        # Group by weather + town + location + cav for unique trajectories
        group_cols = ["town", "location", "cav_id"]
        if "weather" in self.metadata_df.columns:
            group_cols = ["weather"] + group_cols

        for keys, group in self.metadata_df.groupby(group_cols):
            if isinstance(keys, tuple):
                traj_id = "_".join(str(k) for k in keys)
            else:
                traj_id = str(keys)
            trajectories[traj_id] = group.sort_values("frame_id")["sample_idx"].tolist()
        return trajectories

    def split_trajectory_temporal(
        self, indices: List[int], train_ratio: float = 0.7
    ) -> Tuple[List[int], List[int]]:
        """
        Temporal split: first 70% → train, last 30% → test.
        Tests generalization to new positions along trajectory.
        """
        split = int(len(indices) * train_ratio)
        return indices[:split], indices[split:]

    def split_trajectory_random(
        self, indices: List[int], train_ratio: float = 0.7, seed: int = 42
    ) -> Tuple[List[int], List[int]]:
        """
        Random split: shuffle then split 70/30.
        Tests generalization to unseen samples from SAME distribution.
        """
        rng = np.random.RandomState(seed)
        shuffled = np.array(indices.copy())
        rng.shuffle(shuffled)
        split = int(len(shuffled) * train_ratio)
        return shuffled[:split].tolist(), shuffled[split:].tolist()


@dataclass
class ChannelClientData:
    train_indices: List[int]
    test_indices: List[int]
    client_id: int
    trajectory_id: str


def build_clients(
    dataset: ChannelDataset,
    train_ratio: float = 0.7,
    min_trajectory_length: int = 10,
    split_strategy: str = "temporal",  # "temporal" or "random"
) -> Tuple[List[ChannelClientData], Dict[str, List[int]]]:
    """
    Build federated clients from trajectory data.

    Returns
    -------
    clients      : list of ChannelClientData (one per trajectory)
    trajectories : the full trajectory dict — reuse in precompute_beam_changes
                   to avoid rebuilding DatasetSplitter a second time.

    Args:
        split_strategy:
            - "temporal": first 70% train, last 30% test
            - "random"  : random 70/30 split
    """
    splitter = DatasetSplitter(dataset)
    trajectories = splitter.get_trajectory_groups()
    print(f"Total trajectories: {len(trajectories)}")

    trajectories = {
        t: idx for t, idx in trajectories.items() if len(idx) >= min_trajectory_length
    }
    print(f"After length filter (>={min_trajectory_length}): {len(trajectories)}")

    clients = []
    for cid, traj_id in enumerate(sorted(trajectories)):
        if split_strategy == "random":
            train_idx, test_idx = splitter.split_trajectory_random(
                trajectories[traj_id], train_ratio
            )
        else:  # temporal
            train_idx, test_idx = splitter.split_trajectory_temporal(
                trajectories[traj_id], train_ratio
            )
        if not train_idx or not test_idx:
            continue
        clients.append(
            ChannelClientData(train_idx, test_idx, cid, traj_id)
        )

    total_train = sum(len(c.train_indices) for c in clients)
    total_test = sum(len(c.test_indices) for c in clients)
    print(f"\nClients created: {len(clients)} (split_strategy={split_strategy})")
    print(f"Total train samples: {total_train}")
    print(f"Total test samples:  {total_test}")
    print(f"Avg samples per client: {(total_train + total_test) / max(len(clients), 1):.0f}")
    return clients, trajectories

# ── Hierarchical Cluster Assignment ──────────────────────────────────────────
TOWN_ORDER = ["Town03", "Town05", "Town07", "Town10"]


def _client_sort_key(c: ChannelClientData):
    """Sort key: (town_rank, location, weather, cav_id) from trajectory_id."""
    parts    = c.trajectory_id.split("_")
    weather  = parts[0]
    town     = parts[1]
    location = "_".join(parts[2:-1])
    cav_id   = parts[-1]
    return (TOWN_ORDER.index(town) if town in TOWN_ORDER else 99,
            location, weather, cav_id)


def build_clusters(clients: List[ChannelClientData], n_clusters: int = 3
                   ) -> Dict[int, List[int]]:
    """
    Assign FL clients to n_clusters equal-size geographic clusters for
    hierarchical aggregation (Cluster Head model).

    Clients are sorted by (town_rank, location, weather, cav_id) to produce
    a town-ordered list, then divided into n_clusters equal thirds.
    This guarantees equal cluster sizes regardless of per-town location counts.

    Communication design (3 clusters):
        CH0  — Cluster 0 intra-cluster aggregate         (Town03-heavy)
        CH1  — Cluster 1 intra-cluster aggregate          (Town05/07 boundary)
               + global aggregator: receives from CH0 and CH2
        CH2  — Cluster 2 intra-cluster aggregate         (Town10-heavy)

    Returns
    -------
    cluster_map : Dict[cluster_id -> List[client_index]]
        client_index = 0-based position in the original `clients` list.
    """
    sorted_indices = sorted(range(len(clients)), key=lambda i: _client_sort_key(clients[i]))
    N      = len(sorted_indices)
    thirds = [N // n_clusters + (1 if i < N % n_clusters else 0)
              for i in range(n_clusters)]

    cluster_map: Dict[int, List[int]] = {}
    start = 0
    for cid, size in enumerate(thirds):
        cluster_map[cid] = sorted_indices[start : start + size]
        start += size

    # Print summary
    for cid, idxs in cluster_map.items():
        towns_in = [clients[i].trajectory_id.split("_")[1] for i in idxs]
        counts   = {t: towns_in.count(t) for t in TOWN_ORDER if t in towns_in}
        print(f"  Cluster {cid} (CH{cid}): {len(idxs)} clients — {counts}")

    return cluster_map


In [ ]:
# ── Cell: Load Dataset ── channel + multimodal + build clients ────────────────
tf.keras.backend.clear_session()

WEATHER_CONDITIONS = ["sunny", "foggy", "rainy"]
TOWNS = ["Town03", "Town05", "Town07", "Town10"]
DATA_ROOT = "/content/drive/MyDrive/Dataset_flat"

# 1. Channel-only dataset
ds = ChannelDataset(
    root=DATA_ROOT,
    config_name="Nt_1_16_Nr_1_16_fc_28GHz",
    weather_conditions=WEATHER_CONDITIONS,
    towns=TOWNS,
    stride=10,
    Q_tx=CFG["Q_tx"],
    Q_rx=CFG["Q_rx"],
)
print(f"\nChannel dataset: {len(ds)} samples  (stride=10, all towns)")
print(f"Config dirs: {len(ds.config_dirs)}")

x, y = ds[0]
print(f"CSI shape   : {x.shape}")
print(f"Beam index  : {y['beam_index']}  g_opt: {y['g_opt']:.4f}  los: {y['los']}")

# 2. Sensor index (for multimodal)
sensor_idx = SensorIndex(
    root=DATA_ROOT,
    weather_conditions=WEATHER_CONDITIONS,
    towns=TOWNS,
)

# 3. Multimodal dataset (wraps channel dataset + sensor index)
mm_ds = MultimodalChannelDataset(
    channel_dataset=ds,
    sensor_index=sensor_idx,
)

# 4. Build federated clients (trajectory-aware)
clients, _trajectories = build_clients(ds, train_ratio=0.7, min_trajectory_length=10, split_strategy="temporal")

# ── Cap number of clients if MAX_CLIENTS is set ───────────────────────────────
if MAX_CLIENTS is not None:
    clients = clients[:MAX_CLIENTS]
    print(f'\n  Capped to {len(clients)} clients (MAX_CLIENTS={MAX_CLIENTS})')

# ── Precompute beam_change labels ─────────────────────────────────────────────
def precompute_beam_changes(
    channel_ds: ChannelDataset,
    clients: list,
    trajectories: dict,
) -> dict:
    from collections import defaultdict
    import json as _json, hashlib as _hl, io as _io, zipfile as _zf
    print("Precomputing beam_change labels (flat-first batch reads; ZIP fallback)...")
    _t = time.time()

    client_idx_set = set()
    for c in clients:
        client_idx_set.update(c.train_indices)
        client_idx_set.update(c.test_indices)

    needed_indices: set = set()
    for traj_id, indices in trajectories.items():
        if client_idx_set.intersection(indices):
            needed_indices.update(indices)

    # Cache saved to Drive so it survives Colab restarts
    _cache_key  = _hl.md5(str(sorted(needed_indices)).encode()).hexdigest()[:10]
    _cache_file = Path(DATA_ROOT) / f"beam_changes_{_cache_key}.json"
    if _cache_file.exists():
        print(f"  [cache hit] {_cache_file.name}")
        data = _json.load(open(_cache_file))
        changes_cached = {int(k): v for k, v in data.items()}
        for i in client_idx_set:
            changes_cached.setdefault(i, 0)
        n_chg = sum(changes_cached.values())
        print(f"  Done in {time.time()-_t:.1f}s (cached) — beam transitions: {n_chg}/{len(changes_cached)}")
        return changes_cached

    zip_groups = defaultdict(list)
    for idx in needed_indices:
        zip_groups[channel_ds.index[idx].zip_path].append(idx)

    beam_cache: dict = {}
    n_zips = len(zip_groups)
    for z_i, (zip_path, idx_list) in enumerate(zip_groups.items(), 1):
        flat_todo, zip_todo = [], []
        for idx in idx_list:
            fp = zip_path.with_suffix("") / channel_ds.index[idx].inner_npz
            (flat_todo if fp.exists() else zip_todo).append((idx, fp))

        # Flat reads — fall back to ZIP on any error (empty/corrupt file)
        zip_fallback = []
        for idx, fp in flat_todo:
            try:
                arrays = np.load(fp)
                a_sq = np.squeeze(arrays["a"]).astype(np.complex64)
                if a_sq.ndim == 2: a_sq = a_sq[:, :, np.newaxis]
                beam_cache[idx], _, _ = channel_ds.compute_beam_index(a_sq)
            except Exception:
                zip_fallback.append((idx, channel_ds.index[idx].inner_npz))

        # ZIP reads (original + fallback from corrupt flat files)
        all_zip_todo = [(idx, channel_ds.index[idx].inner_npz) for idx, _ in zip_todo] + zip_fallback
        if all_zip_todo:
            try:
                with _zf.ZipFile(zip_path, "r") as zf:
                    for idx, inner in all_zip_todo:
                        try:
                            raw = zf.read(inner)
                            arrays = np.load(_io.BytesIO(raw))
                            a_sq = np.squeeze(arrays["a"]).astype(np.complex64)
                            if a_sq.ndim == 2: a_sq = a_sq[:, :, np.newaxis]
                            beam_cache[idx], _, _ = channel_ds.compute_beam_index(a_sq)
                        except Exception:
                            pass  # skip corrupt ZIP entry
            except (FileNotFoundError, _zf.BadZipFile):
                pass

        if zip_fallback:
            print(f"  [warn] {len(zip_fallback)} corrupt flat files fell back to ZIP")
        print(f"  ZIP {z_i}/{n_zips}: {zip_path.name} "
              f"({len(flat_todo)-len(zip_fallback)} flat + {len(all_zip_todo)} zip)")

    changes: dict = {}
    for traj_id, indices in trajectories.items():
        if not client_idx_set.intersection(indices):
            continue
        prev_beam = None
        for idx in indices:
            b = beam_cache.get(idx)
            if b is None:
                prev_beam = None
                continue
            if idx in client_idx_set:
                changes[idx] = 0 if prev_beam is None else int(b != prev_beam)
            prev_beam = b

    for i in client_idx_set:
        changes.setdefault(i, 0)

    n_chg = sum(changes.values())
    print(f"  Done in {time.time()-_t:.1f}s — {len(needed_indices)} frames across {n_zips} ZIPs | "
          f"beam transitions: {n_chg}/{len(changes)} ({100*n_chg/max(len(changes),1):.1f}%)")
    try:
        _json.dump({str(k): v for k, v in changes.items()}, open(_cache_file, "w"))
        print(f"  [cache saved] → {_cache_file.name}")
    except Exception as e:
        print(f"  [cache save failed] {e}")
    return changes

beam_changes_global = precompute_beam_changes(ds, clients, _trajectories)

# Quick label check
sample_n = min(10, len(ds))
beam_labels = [ds[i][1]["beam_index"].item() for i in range(sample_n)]
gopt_labels = [10.0 * np.log10(max(ds[i][1]["g_opt"].item(), 1e-12))
               for i in range(sample_n)]   # dB
los_labels  = [ds[i][1]["los"].item()        for i in range(sample_n)]
print(f"\nLabel summary (first {sample_n} samples):")
print(f"  beam_index : range [{min(beam_labels)}, {max(beam_labels)}]  "
      f"| unique={len(set(beam_labels))}")
print(f"  g_opt (dB) : mean={np.mean(gopt_labels):.1f}  "
      f"min={min(gopt_labels):.1f}  max={max(gopt_labels):.1f}")
print(f"  los        : LoS={sum(los_labels)}/{sample_n}  "
      f"({100.0*sum(los_labels)/sample_n:.1f}%)")
print(f"  beam_change: transitions in global dict = {sum(beam_changes_global.values())}")
print(f"\nBeam distribution: {np.bincount(beam_labels, minlength=CFG['Q_tx'])}")

# ── Aliases used by the multimodal FL training cell ───────────────────────────
mm_channel_ds  = ds
mm_clients_raw = clients

  [OK] Found channel data: sunny → /content/drive/MyDrive/Dataset_flat/sunny/Channel Data/V2I/Nt_1_16_Nr_1_16_fc_28GHz
  [OK] Found channel data: foggy → /content/drive/MyDrive/Dataset_flat/foggy/Channel Data/Nt_1_16_Nr_1_16_fc_28GHz
  [OK] Found channel data: rainy → /content/drive/MyDrive/Dataset_flat/rainy/Channel Data/V2I/Nt_1_16_Nr_1_16_fc_28GHz

Channel dataset: 11740 samples  (stride=10, all towns)
Config dirs: 3
CSI shape   : (16, 16, 3)
Beam index  : 8  g_opt: 0.0000  los: 0
[SensorIndex] 48 sensor sources indexed (0 zip + 48 flat)
[MultimodalDataset] 11740 samples  |  sensor-aligned: 11140  |  fallback (zeros): 600
Metadata index built: 11740 samples
  Towns: 4, Weather conditions: 3
Total trajectories: 111
After length filter (>=10): 111

Clients created: 111 (split_strategy=temporal)
Total train samples: 8218
Total test samples:  3522
Avg samples per client: 106
Precomputing beam_change labels (flat-first batch reads; ZIP fallback)...
  [cache hit] beam_changes_d32a2128f4.j

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Done in 0.4s (cached) — beam transitions: 1924/11740

Label summary (first 10 samples):
  beam_index : range [8, 8]  | unique=1
  g_opt      : min=0.000000  max=0.000002
  los        : LoS=1/10  (10.0%)
  beam_change: transitions in global dict = 1924

Beam distribution: [ 0  0  0  0  0  0  0  0 10  0  0  0  0  0  0  0]


In [ ]:
# ── Shared FL setup — depends on Cell 12 (dataset + clients) ─────────────────
# Runs once before all experiments.  Every run cell imports from this namespace.
#   mm_client_data : serialisable client specs (train/test indices + traj id)
#   agg_metrics    : Flower weighted-average metric aggregation helper
#   cluster_map_mm : cluster → [client indices] for hierarchical strategies
#   traj_to_idx_mm : trajectory_id → client index (AttackingHierarchicalFedAvg)

mm_client_data = [
    {
        "train_indices": rc.train_indices,
        "test_indices":  rc.test_indices,
        "trajectory_id": rc.trajectory_id,
    }
    for rc in mm_clients_raw
]
print(f"Multimodal clients: {len(mm_client_data)}")

def agg_metrics(metrics):
    agg = {}
    for _, m in metrics:
        for k, v in m.items():
            if isinstance(v, (int, float, np.integer, np.floating)):
                agg.setdefault(k, []).append(float(v))
    return {k: float(np.mean(vs)) for k, vs in agg.items() if vs}

cluster_map_mm = build_clusters(mm_clients_raw)
traj_to_idx_mm = {mm_clients_raw[k].trajectory_id: k
                  for k in range(len(mm_clients_raw))}

print(f"Clusters       : {len(cluster_map_mm)}"
      f"  |  Trajectories: {len(traj_to_idx_mm)}")

In [ ]:
# ── Model Architecture ────────────────────────────────────────────────────────
#  1. ChannelEncoder      — CNN 64→128→256 + SE attention (A2G optimised)
#  2. LiDAREncoder        — PointNet-style (128 pts, 32→64→128)
#  3. IMUEncoder          — GRU(32) over 7 IMU components
#  4. CrossModalAttention — multi-head attention fusion
#  5. BeamModel           — channel-only: 4 task heads (beam / gopt / snr / change)
#  6. MultimodalBeamModel — CSI+LiDAR+IMU: same 4 heads
#
# LoS head REMOVED — binary LoS/NLoS classification removed from all models.
# Rationale: 84% NLOS imbalance, pos_weight tuning unstable, head provides no
# additional information that isn't already captured by gopt (LOS ↔ high gain).
#
# TODO (pending clarification from Dr. Parvej):
#   CSI input augmentation — add summed path gain (scalar per-path powers,
#   summed across MPCs) as an additional channel to the CSI tensor.
#   Current shape: (B, Nr, Nt, 3) → proposed: (B, Nr, Nt, 4) or (B, Nr, Nt, 3+P)
#   where P = number of summed gain channels. Exact shape TBD before implementing.
# ──────────────────────────────────────────────────────────────────────────────

def _compute_normalized_gain(H_batch, pred_indices, tx_codebook, rx_codebook):
    gains = []
    for i in range(len(pred_indices)):
        H           = H_batch[i]
        resp        = rx_codebook.conj().T @ H @ tx_codebook
        gain_matrix = np.abs(resp) ** 2
        achieved    = np.max(gain_matrix[:, pred_indices[i]])
        optimal     = np.max(gain_matrix)
        gains.append(achieved / (optimal + 1e-12))
    return float(np.mean(gains))


@tf.keras.utils.register_keras_serializable()
class SEBlock(tf.keras.layers.Layer):
    """Squeeze-and-Excitation channel attention. Recalibrates CNN maps channel-wise."""
    def __init__(self, channels: int, ratio: int = 4, **kwargs):
        super().__init__(**kwargs)
        self.gap     = keras.layers.GlobalAveragePooling2D()
        self.fc1     = keras.layers.Dense(max(1, channels // ratio), activation="relu")
        self.fc2     = keras.layers.Dense(channels, activation="sigmoid")
        self.reshape = keras.layers.Reshape((1, 1, channels))

    def call(self, x, training=False):
        s = self.gap(x)
        s = self.fc2(self.fc1(s))
        return x * self.reshape(s)


@tf.keras.utils.register_keras_serializable()
class ChannelEncoder(tf.keras.Model):
    """CNN encoder for effective channel matrix. Input: (B, Nr, Nt, 3)."""
    def __init__(self, nr: int, nt: int, emb_dim: int = 128, dropout: float = 0.0, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = keras.layers.Conv2D(64,  3, padding="same", activation="relu")
        self.bn1   = keras.layers.BatchNormalization()
        self.se1   = SEBlock(64)
        self.conv2 = keras.layers.Conv2D(128, 3, padding="same", activation="relu")
        self.bn2   = keras.layers.BatchNormalization()
        self.se2   = SEBlock(128)
        self.conv3 = keras.layers.Conv2D(256, 3, padding="same", activation="relu")
        self.bn3   = keras.layers.BatchNormalization()
        self.se3   = SEBlock(256)
        self.gap   = keras.layers.GlobalAveragePooling2D()
        self.proj  = keras.Sequential([
            keras.layers.Dense(256, activation="relu"),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(emb_dim),
        ])

    def call(self, x, training=False):
        h = self.se1(self.bn1(self.conv1(x), training=training))
        h = self.se2(self.bn2(self.conv2(h), training=training))
        h = self.se3(self.bn3(self.conv3(h), training=training))
        return self.proj(self.gap(h), training=training)


@tf.keras.utils.register_keras_serializable()
class LiDAREncoder(tf.keras.Model):
    """PointNet-style encoder. Input: (B, 128, 3)."""
    def __init__(self, emb_dim: int = 128, dropout: float = 0.0, **kwargs):
        super().__init__(**kwargs)
        self.conv1 = keras.layers.Conv1D(32,  1, activation="relu")
        self.bn1   = keras.layers.BatchNormalization()
        self.conv2 = keras.layers.Conv1D(64,  1, activation="relu")
        self.bn2   = keras.layers.BatchNormalization()
        self.conv3 = keras.layers.Conv1D(128, 1, activation="relu")
        self.bn3   = keras.layers.BatchNormalization()
        self.pool  = keras.layers.GlobalMaxPooling1D()
        self.proj  = keras.Sequential([
            keras.layers.Dense(emb_dim, activation="relu"),
            keras.layers.Dropout(dropout),
        ])

    def call(self, x, training=False):
        h = self.bn1(self.conv1(x), training=training)
        h = self.bn2(self.conv2(h), training=training)
        h = self.bn3(self.conv3(h), training=training)
        return self.proj(self.pool(h), training=training)


@tf.keras.utils.register_keras_serializable()
class IMUEncoder(tf.keras.Model):
    """GRU encoder for IMU vector (7 components). UAV attitude → beam direction."""
    def __init__(self, emb_dim: int = 128, dropout: float = 0.0, **kwargs):
        super().__init__(**kwargs)
        self.gru  = keras.layers.GRU(32, return_sequences=False)
        self.proj = keras.Sequential([
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(emb_dim),
        ])

    def call(self, x, training=False):
        return self.proj(self.gru(tf.expand_dims(x, -1), training=training),
                         training=training)


@tf.keras.utils.register_keras_serializable()
class CrossModalAttention(tf.keras.layers.Layer):
    """Multi-head attention over stacked modality embeddings."""
    def __init__(self, emb_dim: int = 128, n_heads: int = 4, **kwargs):
        super().__init__(**kwargs)
        self.mha  = keras.layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=emb_dim // n_heads)
        self.norm = keras.layers.LayerNormalization()

    def call(self, embeddings, training=False):
        x = tf.stack(embeddings, axis=1)
        return tf.reduce_mean(self.norm(x + self.mha(x, x, training=training)), axis=1)


# ──────────────────────────────────────────────────────────────────────────────
# Channel-only model: 4 task heads (LoS removed)
#   beam  — cross-entropy beam classification
#   gopt  — MSE normalised beamforming gain (z-score)
#   snr   — MSE link SNR proxy (log-normalised gain, z-score) [UAV head]
#   beam_change — BCE beam-switch prediction
# ──────────────────────────────────────────────────────────────────────────────
@tf.keras.utils.register_keras_serializable()
class BeamModel(keras.Model):
    def __init__(self, nr: int, nt: int, beam_codebook_size: int,
                 dropout: float = 0.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder     = ChannelEncoder(nr, nt, emb_dim=128, dropout=dropout)
        self.shared      = keras.Sequential([
            keras.layers.Dense(256, activation="relu"),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(128, activation="relu"),
        ])
        self.beam_head   = keras.layers.Dense(beam_codebook_size, name="beam")
        self.gopt_head   = keras.layers.Dense(1, name="gopt")
        self.snr_head    = keras.layers.Dense(1, name="snr")
        self.change_head = keras.layers.Dense(1, name="beam_change")

    def call(self, x, training=False):
        h = self.shared(self.encoder(x, training=training), training=training)
        return {
            "beam"       : self.beam_head(h),
            "gopt"       : tf.squeeze(self.gopt_head(h),   axis=-1),
            "snr"        : tf.squeeze(self.snr_head(h),    axis=-1),
            "beam_change": tf.squeeze(self.change_head(h), axis=-1),
        }

    def build_model(self, nr, nt):
        _ = self(tf.random.normal((1, nr, nt, 3)), training=False)
        self.built = True


# ──────────────────────────────────────────────────────────────────────────────
# Multimodal model: CSI + LiDAR + IMU, same 4 heads
# ──────────────────────────────────────────────────────────────────────────────
@tf.keras.utils.register_keras_serializable()
class MultimodalBeamModel(keras.Model):
    def __init__(self, nr: int = 16, nt: int = 16, n_beams: int = 16,
                 emb_dim: int = 128, dropout: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.csi_enc     = ChannelEncoder(nr, nt, emb_dim=emb_dim, dropout=dropout)
        self.lidar_enc   = LiDAREncoder(emb_dim=emb_dim, dropout=dropout)
        self.imu_enc     = IMUEncoder(emb_dim=emb_dim, dropout=dropout)
        self.fusion      = CrossModalAttention(emb_dim=emb_dim)
        self.shared      = keras.Sequential([
            keras.layers.Dense(256, activation="relu"),
            keras.layers.BatchNormalization(),
            keras.layers.Dropout(dropout),
            keras.layers.Dense(128, activation="relu"),
        ])
        self.beam_head   = keras.layers.Dense(n_beams, name="beam")
        self.gopt_head   = keras.layers.Dense(1, name="gopt")
        self.snr_head    = keras.layers.Dense(1, name="snr")
        self.change_head = keras.layers.Dense(1, name="beam_change")

    def call(self, csi, lidar, imu, training=False):
        fused = self.fusion([self.csi_enc(csi, training=training),
                             self.lidar_enc(lidar, training=training),
                             self.imu_enc(imu, training=training)], training=training)
        h = self.shared(fused, training=training)
        return {
            "beam"       : self.beam_head(h),
            "gopt"       : tf.squeeze(self.gopt_head(h),   axis=-1),
            "snr"        : tf.squeeze(self.snr_head(h),    axis=-1),
            "beam_change": tf.squeeze(self.change_head(h), axis=-1),
        }

    def build_model(self, nr, nt, n_lidar=128):
        _ = self(tf.random.normal((1, nr, nt, 3)),
                 tf.random.normal((1, n_lidar, 3)),
                 tf.random.normal((1, 7)), training=False)
        self.built = True


print("Models defined (LoS head removed):")
print("  BeamModel          — 4 heads: beam / gopt / snr / beam_change")
print("  MultimodalBeamModel — 4 heads: beam / gopt / snr / beam_change")
print("  ChannelEncoder     — CNN 64→128→256 + SE attention")
print("  TODO: CSI path-gain channel augmentation (pending Dr. Parvej confirmation)")


In [ ]:
# ── HierarchicalFedAvg Strategy ──────────────────────────────────────────────
# Fixes applied (see "Remaining Issues"):
#   [Fix 4] CH_DIVERGENCE_THRESHOLD raised 0.0001 → 0.003: normal intra-cluster
#            divergence was ~0.0008, constantly triggering false S3. Threshold
#            must sit above the noise floor of healthy training.
#   [Fix 6] Divergence population moved BEFORE detection in aggregate_fit so
#            the very round's data is available for S3 logic.
#            len(div_hist) >= 1 (was >= 2) — faster S3 activation.
#            Local-baseline fallback: when other_divs is empty (only-active cluster),
#            compare against own first-5-round min divergence × 3.
#   [S2 REWRITE] Old S2: cosine_sim(CH_now, CH_4rounds_ago) > 0.999 (stagnation).
#                New S2: cosine_sim(CH_now, cluster_member_mean_now) < threshold
#                (drift detection). A Byzantine CH injecting 3-round-old stale
#                weights diverges from current-round peer updates even when its
#                own weights don't look frozen to itself.
#   [DDIL]  Connectivity dropout, Multi-Krum, Reputation scoring (from prior update)

from collections import defaultdict, deque as _deque
from flwr.common import (
    FitRes, EvaluateRes, Parameters,
    parameters_to_ndarrays, ndarrays_to_parameters,
)
from flwr.server.client_proxy import ClientProxy


class HierarchicalFedAvg(fl.server.strategy.FedAvg):

    LOSS_SLOPE_THRESHOLD    = 0.0001
    ISOLATION_ROUNDS        = 4
    REENTRY_FRACS           = [0.30, 0.70, 1.00]
    DETECT_COOLDOWN         = 10
    CH_DIVERGENCE_THRESHOLD = 0.0008  # observed attack div≈0.00190, healthy≈0.00080; gap=0.00110
    # S2: peer-relative gap — fires when this cluster's CH-member sim drops
    # PEER_GAP_THRESHOLD below the mean of other active clusters' sims.
    # Fallback to absolute threshold when only 1 active cluster remains.
    CH_MEMBER_SIM_THRESHOLD = 0.9987   # absolute fallback (single-cluster case)
    S2_PERSISTENT_ROUNDS    = 5      # consecutive rounds (legacy path)
    S2_WINDOW_SIZE          = 7      # sliding-window length
    S2_WINDOW_FIRES         = 3      # fires needed in window to trigger
    PEER_GAP_THRESHOLD      = 0.0005   # peer-relative S2: fire when peer_mean_sim − this_sim > threshold
    REPUTATION_EMA          = 0.7
    REPUTATION_FLOOR        = 0.3
    MULTI_KRUM_F            = 1
    USE_MULTI_KRUM          = True
    DDIL_CONNECTIVITY_PROB  = 0.90

    def __init__(self, cluster_map: Dict[int, List[int]], eval_every: int = 5,
                 ddil_connectivity_prob: float = 0.90,
                 traj_to_idx: Dict[str, int] = None, **kwargs):
        super().__init__(**kwargs)
        self.eval_every             = eval_every
        self.DDIL_CONNECTIVITY_PROB = ddil_connectivity_prob

        self._client_to_cluster: Dict[int, int]       = {}
        self._cluster_to_clients: Dict[int, List[int]] = {}
        for cid, idxs in cluster_map.items():
            self._cluster_to_clients[cid] = list(idxs)
            for idx in idxs:
                self._client_to_cluster[idx] = cid
        self._n_clusters    = len(cluster_map)
        self._total_clients = sum(len(v) for v in cluster_map.values())

        self._ch_assignment: Dict[int, int] = {
            cid: idxs[0] for cid, idxs in cluster_map.items() if idxs
        }
        self._client_signal_hist:    Dict[int, _deque] = defaultdict(lambda: _deque(maxlen=5))
        self._client_weight_hist:    Dict[int, _deque] = defaultdict(lambda: _deque(maxlen=6))
        self._cluster_eval_loss:     Dict[int, _deque] = {cid: _deque(maxlen=5) for cid in cluster_map}
        self._cluster_loss_history:  Dict[int, List[float]] = {cid: [] for cid in cluster_map}
        self._cluster_train_acc_hist: Dict[int, _deque] = {cid: _deque(maxlen=5) for cid in cluster_map}

        self._isolation_state:   Dict[int, int]    = {}
        self._expelled_clients:  set               = set()
        self._reentry_queue:     Dict[int, _deque] = {}
        self._last_detect_round: Dict[int, int]    = {}
        # maxlen=10 so local-baseline fallback has enough history
        self._ch_divergence_hist: Dict[int, _deque] = defaultdict(lambda: _deque(maxlen=10))
        self._s2_consecutive:    Dict[int, int]    = defaultdict(int)
        self._s2_window:         Dict[int, _deque] = defaultdict(lambda: _deque(maxlen=self.S2_WINDOW_SIZE))
        # Track per-cluster CH-member sim history for S2 debugging
        self._ch_member_sim_hist: Dict[int, _deque] = defaultdict(lambda: _deque(maxlen=10))
        self._reputation:        Dict[int, float]  = defaultdict(lambda: 1.0)
        self._missed_rounds:     Dict[int, int]    = defaultdict(int)
        self._events: List[dict] = []

        # ── Trajectory-ID → client-index map (collision-free proxy mapping) ──
        self._traj_to_idx:      Dict[str, int]  = traj_to_idx or {}
        self._proxy_id_map:     Dict[int, int]  = {}   # proxy.cid → client_idx
        self._proxy_map_final:  bool            = False

        # ── Diagnostic: print cluster membership at init ─────────────────────
        print(f"  [CLUSTER INIT] {self._n_clusters} clusters, "
              f"{self._total_clients} total clients")
        for _cid, _idxs in sorted(self._cluster_to_clients.items()):
            _ch = self._ch_assignment.get(_cid, "?")
            _sorted = sorted(_idxs)
            print(f"    C{_cid}: {len(_idxs)} members  CH={_ch}  "
                  f"idx_range=[{_sorted[0]},{_sorted[-1]}]  "
                  f"first5={_sorted[:5]}")

    # ── DDIL connectivity dropout ──────────────────────────────────────────────
    def _ddil_filter(self, results, server_round):
        rng = np.random.default_rng(seed=server_round)
        kept, dropped = [], []
        for proxy, fit_res in results:
            if rng.random() < self.DDIL_CONNECTIVITY_PROB:
                kept.append((proxy, fit_res))
            else:
                client_idx = self._proxy_to_client_idx(proxy)
                self._missed_rounds[client_idx] += 1
                dropped.append(client_idx)
        if dropped:
            print(f"  [DDIL R{server_round}] {len(dropped)} client(s) missed uplink: {dropped}")
        return kept

    # ── Multi-Krum within-cluster ──────────────────────────────────────────────
    @staticmethod
    def _multi_krum(updates: List[Tuple[List[np.ndarray], int]], f: int
                    ) -> List[Tuple[List[np.ndarray], int]]:
        n = len(updates)
        if n <= f + 2 or f == 0:
            return updates
        keep  = n - f
        flats = [np.concatenate([w.flatten() for w in u[0]]) for u in updates]
        dists = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                d = float(np.sum((flats[i] - flats[j]) ** 2))
                dists[i, j] = dists[j, i] = d
        scores   = [float(np.sum(np.sort(dists[i])[1: n - f - 1])) for i in range(n)]
        selected = np.argsort(scores)[:keep]
        return [updates[i] for i in selected]

    # ── Reputation update ──────────────────────────────────────────────────────
    def _update_reputation(self, cluster_id: int, updates: List[Tuple]):
        if len(updates) < 2:
            return
        total_n  = sum(n for _, n in updates)
        mean_vec = sum((np.concatenate([w.flatten() for w in wts]) * (n / total_n))
                       for wts, n in updates)
        members  = self._cluster_to_clients.get(cluster_id, [])
        for cidx, (wts, _) in zip(members, updates):
            flat = np.concatenate([w.flatten() for w in wts])
            sim  = self._cosine_sim(flat, mean_vec)
            self._reputation[cidx] = (self.REPUTATION_EMA * self._reputation[cidx] +
                                      (1.0 - self.REPUTATION_EMA) * sim)

    def configure_evaluate(self, server_round, parameters, client_manager):
        if server_round % self.eval_every != 0:
            return []
        return super().configure_evaluate(server_round, parameters, client_manager)

    def _proxy_to_client_idx(self, proxy) -> int:
        cid = int(proxy.cid)
        if cid in self._proxy_id_map:
            return self._proxy_id_map[cid]
        # Map not yet built — fall back to modulo (pre-round-1 only)
        return cid % self._total_clients

    def _build_proxy_map(self, results) -> None:
        """Populate _proxy_id_map from trajectory_id metadata.
        Run on every round's RAW results (before DDIL filter) until all
        N proxies are registered. Uses trajectory_id for collision-free
        assignment; falls back to next-free-index for any missing entry.
        """
        used = set(self._proxy_id_map.values())
        for proxy, fit_res in results:
            cid = int(proxy.cid)
            if cid in self._proxy_id_map:
                continue
            traj = fit_res.metrics.get("trajectory_id") if hasattr(fit_res, "metrics") else None
            if traj and traj in self._traj_to_idx:
                idx = self._traj_to_idx[traj]
                if idx not in used:
                    self._proxy_id_map[cid] = idx
                    used.add(idx)
                    continue
            # Fallback: assign next free index (no collision)
            for fallback in range(self._total_clients):
                if fallback not in used:
                    self._proxy_id_map[cid] = fallback
                    used.add(fallback)
                    break

    @staticmethod
    def _weighted_avg(weight_list):
        total = sum(n for _, n in weight_list)
        agg   = [sum(w[i] * (n / total) for w, n in weight_list)
                 for i in range(len(weight_list[0][0]))]
        return agg, total

    @staticmethod
    def _flatten(weights) -> np.ndarray:
        return np.concatenate([w.flatten() for w in weights])

    @staticmethod
    def _cosine_sim(a, b) -> float:
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        return float(np.dot(a, b) / (na * nb)) if na > 1e-10 and nb > 1e-10 else 0.0

    def _ch_health_ranking(self, cluster_id: int):
        members = [c for c in self._cluster_to_clients[cluster_id]
                   if c not in self._expelled_clients]
        if not members:
            return []
        raw_lq, raw_ls = {}, {}
        for cidx in members:
            hist = list(self._client_signal_hist[cidx])
            if not hist:
                raw_lq[cidx] = 0.0; raw_ls[cidx] = 0.0
            else:
                # hist entries are (mean_g_opt, mean_beam_change) — LoS head removed
                g  = float(np.mean([h[0] for h in hist]))
                ch = float(np.mean([h[1] for h in hist]))
                raw_lq[cidx] = g
                raw_ls[cidx] = 1.0 - ch   # high stability (low beam change rate) = good CH
        def _norm(d):
            lo, hi = min(d.values()), max(d.values())
            return {k: (v - lo) / (hi - lo + 1e-8) for k, v in d.items()}
        lq_n = _norm(raw_lq); ls_n = _norm(raw_ls)
        scores = {cidx: 0.6 * lq_n[cidx] + 0.4 * ls_n[cidx] for cidx in members}
        return sorted(scores.items(), key=lambda x: -x[1])

    def _detect_suspect(self, cluster_id: int, ch_idx: int, all_sims: dict):
        my_hist = list(self._cluster_eval_loss[cluster_id])
        signal1, slope1 = False, 0.0
        if len(my_hist) >= 3:
            xs     = np.arange(len(my_hist), dtype=float)
            slope1 = float(np.polyfit(xs, my_hist, 1)[0])
            other_slopes = []
            for other_cid, other_hist in self._cluster_eval_loss.items():
                if other_cid == cluster_id: continue
                if other_cid in self._isolation_state or other_cid in self._reentry_queue: continue
                oh = list(other_hist)
                if len(oh) >= 3:
                    other_slopes.append(float(np.polyfit(np.arange(len(oh), dtype=float), oh, 1)[0]))
            if len(other_slopes) >= 1:
                mean_other = float(np.mean(other_slopes))
                signal1 = (slope1 > self.LOSS_SLOPE_THRESHOLD and
                           slope1 > mean_other + self.LOSS_SLOPE_THRESHOLD)

        # ── S2 (REWRITTEN): CH drift from cluster member mean ─────────────────
        # Old S2 compared CH to its own weights 4 rounds ago (stagnation check).
        # That fails for stale-weight injection because 3-round-old weights may
        # still look similar to current weights in a slowly converging model.
        # New S2 compares the CH's current weights to the mean of its cluster
        # members' current weights. A Byzantine CH sending stale updates will
        # diverge from its peers even when it doesn't look frozen to itself.
        # ── S2 (PEER-RELATIVE GAP): use pre-computed sims passed from aggregate_fit ──
        # Peer-gap fires when this cluster's CH-member sim is > PEER_GAP_THRESHOLD
        # below the mean of all other active clusters' sims.
        # This prevents false positives when ALL clusters simultaneously have low
        # sim during convergence dips — only the outlier cluster gets flagged.
        sim = all_sims.get(cluster_id, float('nan'))
        signal2 = False
        if sim == sim:  # not nan
            self._ch_member_sim_hist[cluster_id].append(sim)
            active_peer_sims = [
                s for cid, s in all_sims.items()
                if cid != cluster_id
                and cid not in self._isolation_state
                and cid not in self._reentry_queue
                and s == s  # exclude nan
            ]
            if len(active_peer_sims) >= 1:
                peer_mean = float(np.mean(active_peer_sims))
                gap = peer_mean - sim
                signal2 = gap > self.PEER_GAP_THRESHOLD
            else:
                # Only 1 active cluster — fall back to absolute threshold
                signal2 = sim < self.CH_MEMBER_SIM_THRESHOLD

        if signal2:
            self._s2_consecutive[cluster_id] += 1
        else:
            self._s2_consecutive[cluster_id] = 0
        self._s2_window[cluster_id].append(0 if (isinstance(sim, float) and (sim != sim)) else int(signal2))

        # ── S3: CH diverges from cluster members ──────────────────────────────
        # Fix [6a]: require len >= 1 (was >= 2) — activates one round earlier
        # Fix [6b]: local-baseline fallback when all other clusters are isolated/absent
        my_div_hist = list(self._ch_divergence_hist[cluster_id])
        signal3, divergence = False, 0.0
        other_divs = []
        if len(my_div_hist) >= 1:
            divergence = float(my_div_hist[-1])
            other_divs = [
                float(np.mean(list(self._ch_divergence_hist[c])))
                for c in range(self._n_clusters)
                if c != cluster_id
                and len(self._ch_divergence_hist[c]) >= 1
                and c not in self._isolation_state
                and c not in self._reentry_queue
            ]
            if len(other_divs) >= 1:
                signal3 = divergence > float(np.mean(other_divs)) + self.CH_DIVERGENCE_THRESHOLD
            elif len(my_div_hist) >= 5:
                baseline = float(np.min(my_div_hist[:5]))
                signal3  = divergence > baseline * 3.0 + self.CH_DIVERGENCE_THRESHOLD
                if signal3:
                    print(f"    [S3-LOCAL C{cluster_id}] div={divergence:.5f} "
                          f"baseline={baseline:.5f} (no peer clusters for comparison)")
            else:
                signal3 = False

        s2_window_count = sum(self._s2_window[cluster_id])
        s2_persistent = (self._s2_consecutive[cluster_id] >= self.S2_PERSISTENT_ROUNDS
                         or s2_window_count >= self.S2_WINDOW_FIRES)
        other_divs_str = " ".join(f"{x:.5f}" for x in other_divs) if other_divs else "n/a"
        sim_str = f"{sim:.6f}" if sim == sim else "nan(no_members)"
        print(f"    [DBG C{cluster_id}] CH_member_sim={sim_str} div={divergence:.5f} "
              f"other_divs=[{other_divs_str}] S2={signal2}(×{self._s2_consecutive[cluster_id]} win={s2_window_count}/{self.S2_WINDOW_SIZE}) "
              f"S3={signal3} s2_persist={s2_persistent}")

        reason = (f"loss_slope={slope1:.5f} ch_member_sim={sim:.6f} "
                  f"ch_divergence={divergence:.5f} S1={signal1} S2={signal2} S3={signal3} "
                  f"s2_persist={s2_persistent}")
        return (signal2 and (signal3 or s2_persistent)), reason

    def _re_elect(self, cluster_id: int):
        old_ch = self._ch_assignment[cluster_id]
        self._expelled_clients.add(old_ch)
        self._cluster_to_clients[cluster_id] = [
            c for c in self._cluster_to_clients[cluster_id] if c != old_ch]
        self._s2_consecutive[cluster_id] = 0
        self._s2_window[cluster_id].clear()  # FIX: clear stale window on re-election
        print(f"  [EXPEL]    cluster={cluster_id}: CH {old_ch} expelled")
        ranked = self._ch_health_ranking(cluster_id)
        if ranked:
            new_ch, score = ranked[0]
            self._ch_assignment[cluster_id] = new_ch
            print(f"  [RE-ELECT] cluster={cluster_id}: new CH={new_ch} (health={score:.3f})")
        else:
            print(f"  [RE-ELECT] cluster={cluster_id}: no members left")
            self._ch_assignment.pop(cluster_id, None)
        self._isolation_state[cluster_id] = self.ISOLATION_ROUNDS
        print(f"  [ISOLATE]  cluster={cluster_id}: {self.ISOLATION_ROUNDS} rounds")

    def aggregate_evaluate(self, server_round, results, failures):
        cluster_losses: Dict[int, List[float]] = defaultdict(list)
        for proxy, ev_res in results:
            client_idx = self._proxy_to_client_idx(proxy)
            if client_idx in self._expelled_clients: continue
            cid = self._client_to_cluster.get(client_idx, 0)
            cluster_losses[cid].append(float(ev_res.loss))
        for cid, losses in cluster_losses.items():
            mean_loss = float(np.mean(losses))
            self._cluster_eval_loss[cid].append(mean_loss)
            self._cluster_loss_history[cid].append(mean_loss)
        return super().aggregate_evaluate(server_round, results, failures)

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        # ── Build proxy→index map (runs on raw results before DDIL) ──────────
        # Processes ALL proxies so DDIL-dropped clients also get registered.
        if not self._proxy_map_final:
            self._build_proxy_map(results)
            if len(self._proxy_id_map) >= self._total_clients:
                self._proxy_map_final = True
                print(f"  [PROXY MAP] Fully built at round {server_round} "
                      f"({len(self._proxy_id_map)} entries, "
                      f"traj_id_hits={sum(1 for k in self._proxy_id_map)}, "
                      f"coverage=100%)")

        # ── Permanently blacklist expelled clients from contributing ──────────
        # Do this BEFORE DDIL filter so expelled clients don't count against
        # drop statistics and can never update weight/signal histories.
        _pre_expel = len(results)
        results = [(p, r) for p, r in results
                   if self._proxy_to_client_idx(p) not in self._expelled_clients]
        if len(results) < _pre_expel:
            print(f"  [EXPELLED FILTER R{server_round}] "
                  f"removed {_pre_expel - len(results)} expelled client update(s)")

        # DDIL: drop clients that fail to connect this round
        results = self._ddil_filter(results, server_round)
        if not results:
            print(f"  [Round {server_round}] all clients dropped by DDIL — skipping")
            return None, {}

        # ── Build result map FIRST (needed for divergence population) ──────────
        # [Fix 6]: result map and divergence must be populated BEFORE detection
        # so that _detect_suspect has this round's data available.
        _result_map: Dict[int, Tuple] = {}
        for _p, _fr in results:
            _cidx = self._proxy_to_client_idx(_p)
            _result_map[_cidx] = (parameters_to_ndarrays(_fr.parameters), _fr.num_examples)

        # ── Populate CH divergence (BEFORE detection) ─────────────────────────
        for cid in range(self._n_clusters):
            ch_idx_s3 = self._ch_assignment.get(cid)
            if ch_idx_s3 is None or ch_idx_s3 in self._expelled_clients: continue
            ch_entry = _result_map.get(ch_idx_s3)
            if ch_entry is None: continue
            ch_flat = self._flatten(ch_entry[0])
            non_ch  = [(w, n) for cidx, (w, n) in _result_map.items()
                       if cidx in self._cluster_to_clients.get(cid, [])
                       and cidx != ch_idx_s3 and cidx not in self._expelled_clients]
            if len(non_ch) >= 1:
                avg_excl, _ = self._weighted_avg(non_ch)
                div = 1.0 - self._cosine_sim(ch_flat, self._flatten(avg_excl))
                self._ch_divergence_hist[cid].append(div)

        # ── Update weight/signal histories ────────────────────────────────────
        for proxy, fit_res in results:
            client_idx = self._proxy_to_client_idx(proxy)
            if client_idx in self._expelled_clients: continue
            m = fit_res.metrics
            # LoS head removed — signal hist is now (mean_g_opt, mean_beam_change)
            if "mean_g_opt" in m:
                self._client_signal_hist[client_idx].append(
                    (m["mean_g_opt"], m["mean_beam_change"]))
            self._client_weight_hist[client_idx].append(
                self._flatten(parameters_to_ndarrays(fit_res.parameters)))

        # ── Proxy→index diagnostic (rounds 1–3) ──────────────────────────────
        if server_round <= 3:
            _seen = set(self._proxy_to_client_idx(p) for p, _ in results)
            for _cid in range(self._n_clusters):
                _members = set(self._cluster_to_clients.get(_cid, []))
                _with_hist = {c for c in _members if self._client_weight_hist[c]}
                _in_round  = _members & _seen
                print(f"    [IDX-DBG R{server_round} C{_cid}] "
                      f"members={len(_members)}  "
                      f"in_round={len(_in_round)}  "
                      f"have_hist={len(_with_hist)}  "
                      f"missing_hist={sorted(_members - _with_hist)[:8]}")
            if server_round == 1:
                _sample = [(p.cid, self._proxy_to_client_idx(p))
                           for p, _ in list(results)[:6]]
                print(f"    [PROXY-MAP R1] cid→idx samples: {_sample}")

        # ── Health display (train_beam_acc + loss from fit metrics) ───────────
        health_parts = []
        for cid in range(self._n_clusters):
            members = [c for c in self._cluster_to_clients.get(cid, [])
                       if c not in self._expelled_clients]
            fit_map = {self._proxy_to_client_idx(p): r.metrics
                       for p, r in results if self._proxy_to_client_idx(p) in members}
            if fit_map:
                accs   = [m.get("train_beam_acc") or m.get("beam_accuracy") or 0.0
                          for m in fit_map.values()]
                losses = [m.get("loss", 0.0) for m in fit_map.values()]
                health_parts.append(
                    f"C{cid}[tr_acc={np.mean(accs):.3f} loss={np.mean(losses):.3f}]")
        if health_parts:
            print(f"    Health R{server_round}: {' '.join(health_parts)}")

        for cid in range(self._n_clusters):
            members = [c for c in self._cluster_to_clients.get(cid, [])
                       if c not in self._expelled_clients]
            accs = [fit_res.metrics.get("train_beam_acc") or fit_res.metrics.get("beam_accuracy") or 0.0
                    for proxy, fit_res in results
                    if self._proxy_to_client_idx(proxy) in members]
            if accs:
                self._cluster_train_acc_hist[cid].append(float(np.mean(accs)))

        # ── Pre-compute per-cluster CH-member sims for peer-gap S2 ─────────────
        # Done ONCE here so every cluster's detection sees the same snapshot.
        _cluster_sims: Dict[int, float] = {}
        for _scid in range(self._n_clusters):
            _sch = self._ch_assignment.get(_scid)
            if _sch is None or _sch in self._expelled_clients: continue
            _swh = self._client_weight_hist[_sch]
            if not _swh: continue
            _sch_flat = _swh[-1]
            _smember_flats = [
                self._client_weight_hist[c][-1]
                for c in self._cluster_to_clients.get(_scid, [])
                if c != _sch and c not in self._expelled_clients and self._client_weight_hist[c]
            ]
            if len(_smember_flats) >= 1:
                _smem_mean = np.mean(np.stack(_smember_flats), axis=0)
                _cluster_sims[_scid] = self._cosine_sim(_sch_flat, _smem_mean)

        # ── Byzantine detection (now has current-round divergence) ────────────
        for cid in range(self._n_clusters):
            ch_idx = self._ch_assignment.get(cid)
            if (ch_idx is None or cid in self._isolation_state or cid in self._reentry_queue):
                continue
            if server_round - self._last_detect_round.get(cid, 0) < self.DETECT_COOLDOWN:
                continue
            is_suspect, reason = self._detect_suspect(cid, ch_idx, _cluster_sims)
            if is_suspect:
                print(f"  [DETECT R{server_round}] cluster={cid} CH={ch_idx}: {reason}")
                self._events.append({"round": server_round, "event": "detect", "cluster": cid})
                self._last_detect_round[cid] = server_round
                self._re_elect(cid)

        # ── Cluster aggregation (Multi-Krum + reputation weighting) ──────────
        cluster_updates: Dict[int, List[Tuple]] = {k: [] for k in range(self._n_clusters)}
        for proxy, fit_res in results:
            client_idx = self._proxy_to_client_idx(proxy)
            if client_idx in self._expelled_clients: continue
            cluster_id = self._client_to_cluster.get(client_idx, 0)
            cluster_updates[cluster_id].append(
                (parameters_to_ndarrays(fit_res.parameters), fit_res.num_examples))

        cluster_aggs, active_cids = [], []
        for cid in range(self._n_clusters):
            updates = cluster_updates[cid]
            if not updates: continue

            if self.USE_MULTI_KRUM and len(updates) > self.MULTI_KRUM_F + 2:
                updates_krum = self._multi_krum(updates, self.MULTI_KRUM_F)
                if len(updates_krum) < len(updates):
                    print(f"  [KRUM C{cid}] removed {len(updates)-len(updates_krum)} divergent update(s)")
                updates = updates_krum

            # Build ordered client-idx list matching the update insertion order
            _cid_order = [self._proxy_to_client_idx(p)
                          for p, _ in results
                          if (self._proxy_to_client_idx(p) not in self._expelled_clients
                              and self._client_to_cluster.get(
                                  self._proxy_to_client_idx(p), -1) == cid)]
            rep_updates = []
            for i, (wts, n) in enumerate(updates):
                cidx = _cid_order[i] if i < len(_cid_order) else -1
                rep = max(self.REPUTATION_FLOOR, self._reputation[cidx])
                rep_updates.append((wts, max(1, int(n * rep))))
            self._update_reputation(cid, updates)

            agg, total = self._weighted_avg(rep_updates)
            if cid in self._isolation_state:
                print(f"  [ISOLATED] cluster={cid}: {self._isolation_state[cid]} rounds remaining")
                continue
            if cid in self._reentry_queue and self._reentry_queue[cid]:
                total = max(1, int(total * self._reentry_queue[cid][0]))
            cluster_aggs.append((agg, total))
            active_cids.append(cid)

        for cid in list(self._isolation_state.keys()):
            self._isolation_state[cid] -= 1
            if self._isolation_state[cid] <= 0:
                del self._isolation_state[cid]
                self._reentry_queue[cid] = _deque(self.REENTRY_FRACS)
                self._events.append({"round": server_round, "event": "reentry_start", "cluster": cid})
                print(f"  [→REENTRY] cluster={cid}: beginning graduated re-entry")

        for cid in list(self._reentry_queue.keys()):
            frac = self._reentry_queue[cid].popleft()
            print(f"  [REENTRY]  cluster={cid}: participation={frac:.0%}")
            if not self._reentry_queue[cid]:
                del self._reentry_queue[cid]
                print(f"  [REENTRY]  cluster={cid}: fully re-integrated")

        if not cluster_aggs:
            print(f"  [Round {server_round}] all clusters isolated — global model unchanged")
            return None, {}

        global_weights, grand_total = self._weighted_avg(cluster_aggs)
        labels = [f"CH{cid}←{n}" for cid, (_, n) in zip(active_cids, cluster_aggs)]
        print(f"  [Round {server_round}] {' '.join(labels)} → global ({grand_total} total)")
        return ndarrays_to_parameters(global_weights), {}


print(f"HierarchicalFedAvg defined.")
print(f"  CH_DIVERGENCE_THRESHOLD  = {HierarchicalFedAvg.CH_DIVERGENCE_THRESHOLD} (raised — fixes false S3)")
print(f"  CH_MEMBER_SIM_THRESHOLD  = {HierarchicalFedAvg.CH_MEMBER_SIM_THRESHOLD} (S2 drift: fires when CH < peers)")
print(f"  S2 REWRITTEN: CH vs cluster member mean (not self 4-rounds-ago)")
print(f"  S3: len(div_hist) >= 1 now; local-baseline fallback when no peers")
print(f"  Divergence populated BEFORE detection (same-round data available)")
print(f"  Health display: checks 'train_beam_acc' OR 'beam_accuracy' key")
print(f"  Signal hist: (mean_g_opt, mean_beam_change) — LoS head removed")

In [ ]:
# ── AttackingHierarchicalFedAvg — Byzantine attack injection ─────────────────

class AttackingHierarchicalFedAvg(HierarchicalFedAvg):
    """HierarchicalFedAvg + configurable Byzantine attack on a target cluster.

    attack_mode:
        "stale"    — inject 3-round-old weights (default, proven detectable)
        "poison"   — negate weights (sign-flip; stronger gradient corruption)
        "adaptive" — stale injection + re-targets new CH after expulsion
    """

    def __init__(self, attack_cluster: int, attack_start_round: int,
                 attack_mode: str = "stale", **kwargs):
        super().__init__(**kwargs)
        self._attack_cluster     = attack_cluster
        self._attack_start_round = max(4, attack_start_round)
        self._attack_mode        = attack_mode.lower()
        self._ch_actual_cache    = _deque(maxlen=6)

        # Initial CH target locked to cluster's first CH.
        ch_members = self._cluster_to_clients.get(attack_cluster, [])
        self._attack_ch_idx = ch_members[0] if ch_members else None
        print(f"  [ATTACK MODE] {self._attack_mode}  target_cluster={attack_cluster}  "
              f"initial_ch={self._attack_ch_idx}  start_round={self._attack_start_round}")

    def _get_current_attack_target(self) -> int:
        """In adaptive mode, always follow the current elected CH."""
        if self._attack_mode == "adaptive":
            return self._ch_assignment.get(self._attack_cluster, self._attack_ch_idx)
        return self._attack_ch_idx

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        ch_idx = self._get_current_attack_target()

        # ── Round 1 verification: confirm attack target matches elected CH ─────
        if server_round == 1 and ch_idx is not None:
            elected_ch = self._ch_assignment.get(self._attack_cluster)
            match = "✓ MATCH" if ch_idx == elected_ch else "✗ MISMATCH"
            print(f"  [ATTACK INIT] attack_cluster={self._attack_cluster} "
                  f"attack_ch_idx={ch_idx} elected_ch={elected_ch} {match}")

        results_list = list(results)

        # ── Stop injection if target is expelled or no longer the elected CH ──
        # Stale/poison modes: attack ends permanently once CH is dethroned.
        # Adaptive mode: _get_current_attack_target() already follows the new CH.
        if self._attack_mode != "adaptive":
            elected_ch = self._ch_assignment.get(self._attack_cluster)
            if (ch_idx is None
                    or ch_idx in self._expelled_clients
                    or ch_idx != elected_ch):
                if server_round >= self._attack_start_round:
                    print(f"  [ATTACK HALTED R{server_round}] "
                          f"CH {ch_idx} expelled/replaced (now CH={elected_ch}). "
                          f"Injection stopped.")
                return super().aggregate_fit(server_round, results_list, failures)

        for i, (proxy, fit_res) in enumerate(results_list):
            client_idx = self._proxy_to_client_idx(proxy)
            if client_idx != ch_idx:
                continue

            real_weights = parameters_to_ndarrays(fit_res.parameters)
            self._ch_actual_cache.append(real_weights)

            if server_round < self._attack_start_round:
                break  # attack not yet active

            if self._attack_mode in ("stale", "adaptive"):
                if len(self._ch_actual_cache) < 4:
                    break
                injected = self._ch_actual_cache[-4]
                mode_label = "stale-3"

            elif self._attack_mode == "poison":
                # Sign-flip all weight arrays (strong gradient inversion)
                injected   = [-w for w in real_weights]
                mode_label = "sign-flip"

            else:
                break  # unknown mode — no-op

            poisoned_params = ndarrays_to_parameters(injected)
            from flwr.common import FitRes as _FitRes
            results_list[i] = (proxy, _FitRes(
                status       = fit_res.status,
                parameters   = poisoned_params,
                num_examples = fit_res.num_examples,
                metrics      = fit_res.metrics,
            ))
            print(f"  [ATTACK R{server_round}] mode={mode_label} cluster={self._attack_cluster} "
                  f"CH={ch_idx}: weights injected")
            break

        return super().aggregate_fit(server_round, results_list, failures)


print("AttackingHierarchicalFedAvg defined.")
print("  attack_mode: 'stale' (default) | 'poison' (sign-flip) | 'adaptive' (re-target)")
print("  Adaptive mode tracks new CH after expulsion — sustained attack across re-elections.")


In [ ]:
# ── MultimodalBeamFlowerClient ────────────────────────────────────────────────
# Modalities: CSI + LiDAR-128 + IMU-GRU
# Multi-task: beam_index | g_opt | beam_change   (LoS removed)
# Module-level actor cache — persists across FL rounds within same Ray worker.

_MM_ACTOR_CACHE: dict = {}   # {sample_idx: (csi, lidar, imu, labels)}


class MultimodalBeamFlowerClient(fl.client.NumPyClient):

    LOSS_WEIGHTS = {"beam": 1.0, "gopt": 1.5, "change": 0.5}

    def __init__(self, model, dataset, train_indices, test_indices,
                 cfg, trajectory_id, beam_changes: dict):
        self.model          = model
        self.dataset        = dataset
        self.train_indices  = train_indices
        self.test_indices   = test_indices
        self.cfg            = cfg
        self.trajectory_id  = trajectory_id
        self.beam_changes   = beam_changes
        self.optimizer      = keras.optimizers.Adam(cfg["lr"])
        self.grad_clip_norm = cfg.get("grad_clip_norm", 5.0)
        self.batch_size     = cfg.get("batch_size", 64)
        self._gopt_stats    = None
        self.tx_codebook    = None
        self.rx_codebook    = None

    def _load(self, indices, return_H=False, split="train"):
        """Lazy-load multimodal data via module-level actor cache.
        Returns (csi, lidar, imu, beam, gopt_znorm, change [, H])."""
        global _MM_ACTOR_CACHE
        missing = [i for i in indices if i not in _MM_ACTOR_CACHE]
        for i in missing:
            csi, lidar, imu, labels = self.dataset[i]
            _MM_ACTOR_CACHE[i] = (csi, lidar, imu, labels)

        CSI, LIDAR, IMU = [], [], []
        y_beam, y_gopt, y_change, Hs = [], [], [], []
        for i in indices:
            csi, lidar, imu, labels = _MM_ACTOR_CACHE[i]
            CSI.append(csi); LIDAR.append(lidar); IMU.append(imu)
            y_beam.append(int(labels["beam_index"]))
            y_gopt.append(float(labels.get("g_opt", 1.0)))
            y_change.append(int(self.beam_changes.get(i, 0)))
            if return_H:
                Hs.append(labels["H_complex"])

        csi_arr   = np.stack(CSI).astype(np.float32)
        lidar_arr = np.stack(LIDAR).astype(np.float32)
        imu_arr   = np.stack(IMU).astype(np.float32)

        mean = csi_arr.mean(axis=(1, 2), keepdims=True)
        std  = csi_arr.std(axis=(1, 2),  keepdims=True) + 1e-8
        csi_arr = (csi_arr - mean) / std

        # ── gopt: dB conversion + z-normalise ───────────────────────────────
        # Raw beamforming gain is ~1e-5 scale.  Converting to dB first maps it
        # to a stable range (−120 … 0 dB) before z-normalisation.
        # This also fixes cross-worker NMSE explosion: Flower's Ray engine may
        # run evaluate() on a different actor than fit(), so self._gopt_stats
        # can be None on a fresh worker.  The dB local-fallback ensures y_gopt
        # and gopt_preds are always on the same O(1) scale.
        gopt_raw = np.array(y_gopt, dtype=np.float32)
        gopt_db  = (10.0 * np.log10(np.clip(gopt_raw, 1e-12, None))).astype(np.float32)

        if split == "train":
            # Store train-set dB statistics for consistent normalisation
            self._gopt_stats = (float(gopt_db.mean()), float(gopt_db.std()) + 1e-8)

        if self._gopt_stats is not None:
            mu_g, sigma_g = self._gopt_stats
        else:
            # Cross-worker fallback: normalise by this batch's own dB stats.
            # Values remain O(1) so NMSE never explodes even on a fresh worker.
            mu_g    = float(gopt_db.mean())
            sigma_g = float(gopt_db.std()) + 1e-8

        gopt_arr = (gopt_db - mu_g) / sigma_g

        result = (
            csi_arr, lidar_arr, imu_arr,
            np.array(y_beam,   dtype=np.int32),
            gopt_arr,
            np.array(y_change, dtype=np.int32),
        )
        if return_H:
            result = result + (np.stack(Hs),)
        return result

    def _ensure_built(self):
        if self.model.built:
            return
        idx = self.train_indices[0] if self.train_indices else 0
        if idx not in _MM_ACTOR_CACHE:
            csi, lidar, imu, labels = self.dataset[idx]
            _MM_ACTOR_CACHE[idx] = (csi, lidar, imu, labels)
        csi, lidar, imu, _ = _MM_ACTOR_CACHE[idx]
        self.model(
            tf.constant(csi[np.newaxis],   tf.float32),
            tf.constant(lidar[np.newaxis], tf.float32),
            tf.constant(imu[np.newaxis],   tf.float32),
            training=False,
        )
        self.model.built = True

    def get_parameters(self, config):
        self._ensure_built()
        return self.model.get_weights()

    def fit(self, parameters, config):
        self._ensure_built()
        self.model.set_weights(parameters)
        csi_arr, lidar_arr, imu_arr, y_beam, y_gopt, y_change = self._load(
            self.train_indices, split="train")

        w         = self.LOSS_WEIGHTS
        n, bs     = len(csi_arr), self.batch_size
        smooth    = self.cfg.get("label_smoothing", 0.0)
        pw_change = self.cfg.get("pos_weight_change", 1.0)
        ep_losses = []

        for _ in range(self.cfg["local_epochs"]):
            perm = np.random.permutation(n)
            batch_losses = []
            for start in range(0, n, bs):
                end      = min(start + bs, n)
                idx      = perm[start:end]
                csi_b    = tf.constant(csi_arr[idx],                     tf.float32)
                lidar_b  = tf.constant(lidar_arr[idx],                   tf.float32)
                imu_b    = tf.constant(imu_arr[idx],                     tf.float32)
                beam_b   = tf.constant(y_beam[idx],                      tf.int32)
                gopt_b   = tf.constant(y_gopt[idx],                      tf.float32)
                change_b = tf.constant(y_change[idx].astype(np.float32), tf.float32)

                with tf.GradientTape() as tape:
                    outs  = self.model(csi_b, lidar_b, imu_b, training=True)
                    n_cls = outs["beam"].shape[-1]
                    if smooth > 0.0:
                        oh            = tf.one_hot(beam_b, n_cls)
                        smooth_labels = oh * (1.0 - smooth) + smooth / tf.cast(n_cls, tf.float32)
                        loss_beam     = tf.reduce_mean(
                            tf.nn.softmax_cross_entropy_with_logits(
                                labels=smooth_labels, logits=outs["beam"]))
                    else:
                        loss_beam = tf.reduce_mean(
                            tf.nn.sparse_softmax_cross_entropy_with_logits(
                                labels=beam_b, logits=outs["beam"]))

                    loss_gopt   = tf.reduce_mean(tf.square(outs["gopt"] - gopt_b))
                    loss_change = tf.reduce_mean(
                        tf.nn.weighted_cross_entropy_with_logits(
                            labels=change_b, logits=outs["beam_change"],
                            pos_weight=pw_change))

                    loss = (w["beam"]   * loss_beam   +
                            w["gopt"]   * loss_gopt   +
                            w["change"] * loss_change)

                grads = tape.gradient(loss, self.model.trainable_weights)
                safe  = [g if g is not None else tf.zeros_like(v)
                         for g, v in zip(grads, self.model.trainable_weights)]
                clipped, _ = tf.clip_by_global_norm(safe, self.grad_clip_norm)
                self.optimizer.apply_gradients(zip(clipped, self.model.trainable_weights))
                batch_losses.append(float(loss.numpy()))
            ep_losses.append(np.mean(batch_losses))

        # Train beam accuracy
        try:
            preds = []
            for s in range(0, n, bs):
                logits = self.model(
                    tf.constant(csi_arr[s:min(s+bs,n)],   tf.float32),
                    tf.constant(lidar_arr[s:min(s+bs,n)], tf.float32),
                    tf.constant(imu_arr[s:min(s+bs,n)],   tf.float32),
                    training=False)["beam"]
                preds.append(np.argmax(logits.numpy(), axis=1))
            train_beam_acc = float(np.mean(
                np.concatenate(preds).astype(np.int64) == np.asarray(y_beam, np.int64)))
        except Exception as e:
            train_beam_acc = 0.0

        return self.model.get_weights(), len(self.train_indices), {
            "loss"             : ep_losses[-1],
            "train_beam_acc"   : train_beam_acc,
            "trajectory_id"    : self.trajectory_id,
            "mean_g_opt"       : float(self._gopt_stats[0]) if self._gopt_stats else 0.0,
            "mean_beam_change" : float(np.mean(y_change.astype(np.float32))),
        }

    def evaluate(self, parameters, config):
        self._ensure_built()
        self.model.set_weights(parameters)
        csi_arr, lidar_arr, imu_arr, y_beam, y_gopt, y_change, H_test = self._load(
            self.test_indices, return_H=True, split="test")

        bs, all_outs = self.batch_size, []
        for start in range(0, len(csi_arr), bs):
            end  = min(start + bs, len(csi_arr))
            outs = self.model(
                tf.constant(csi_arr[start:end],   tf.float32),
                tf.constant(lidar_arr[start:end],  tf.float32),
                tf.constant(imu_arr[start:end],   tf.float32),
                training=False,
            )
            all_outs.append({k: v.numpy() for k, v in outs.items()})

        beam_logits   = np.concatenate([o["beam"]        for o in all_outs])
        gopt_preds    = np.concatenate([o["gopt"]        for o in all_outs])
        change_logits = np.concatenate([o["beam_change"] for o in all_outs])

        y_tf      = tf.constant(y_beam, dtype=tf.int32)
        logits_tf = tf.constant(beam_logits, dtype=tf.float32)
        loss_beam = float(tf.reduce_mean(
            tf.nn.sparse_softmax_cross_entropy_with_logits(
                labels=y_tf, logits=logits_tf)).numpy())
        preds    = np.argmax(beam_logits, axis=1)
        acc      = float(np.mean(preds == np.asarray(y_beam, np.int64)))
        top3_acc = float(tf.reduce_mean(
            tf.cast(tf.math.in_top_k(targets=y_tf, predictions=logits_tf, k=3),
                    tf.float32)).numpy())

        eval_n    = min(len(preds), 200)
        norm_gain = _compute_normalized_gain(
            H_test[:eval_n], preds[:eval_n],
            self.tx_codebook if self.tx_codebook is not None else self.dataset.tx_codebook,
            self.rx_codebook if self.rx_codebook is not None else self.dataset.rx_codebook)

        gopt_err      = gopt_preds - y_gopt
        gopt_mae_znorm = float(np.mean(np.abs(gopt_err)))        # z-norm (for loss)
        _sigma_g      = float(self._gopt_stats[1]) if self._gopt_stats is not None else 1.0
        gopt_mae      = gopt_mae_znorm * _sigma_g                # dB (for reporting)

        change_preds = (change_logits > 0).astype(np.int32)
        change_acc   = float(np.mean(change_preds == y_change))

        w = self.LOSS_WEIGHTS
        combined_loss = (w["beam"]   * loss_beam        +
                         w["gopt"]   * gopt_mae_znorm   +
                         w["change"] * (1.0 - change_acc))

        return float(combined_loss), len(self.test_indices), {
            "beam_accuracy"        : acc,
            "beam_top3_accuracy"   : top3_acc,
            "normalized_gain"      : norm_gain,
            "beam_num_unique_preds": int(len(np.unique(preds))),
            "gopt_mae"             : gopt_mae,   # dB
            "beam_change_accuracy" : change_acc,
            "loss"                 : combined_loss,
        }


print("MultimodalBeamFlowerClient defined.")
print(f"  Heads   : beam | gopt | beam_change  (LoS removed)")
print(f"  Weights : {MultimodalBeamFlowerClient.LOSS_WEIGHTS}")

In [ ]:
# ── Run 3: Multimodal Hierarchical FL + Byzantine Attack (CH0) ───────────────
# Same multimodal hierarchical setup as Run 2, but with stale-weight Byzantine
# injection targeting Cluster 0's cluster head starting from round 11.
# Tests the S2 (CH-member drift) and S3 (inter-cluster divergence) detectors.

if ray.is_initialized():
    ray.shutdown()
    print("Ray shutdown")

_nr3      = mm_channel_ds.nr
_nt3      = mm_channel_ds.nt
_n_beams3 = CFG["Q_tx"]

MM_ATTACK_CLUSTER     = 0
MM_ATTACK_START_ROUND = 11
MM_ROUNDS             = 100

def mm_client_fn(context: fl.common.Context) -> fl.client.Client:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    client_idx = int(context.node_id) % len(mm_client_data)
    data = mm_client_data[client_idx]
    model = MultimodalBeamModel(nr=_nr3, nt=_nt3, n_beams=_n_beams3, dropout=0.1)
    _client = MultimodalBeamFlowerClient(
        model=model, dataset=mm_ds,
        train_indices=data["train_indices"], test_indices=data["test_indices"],
        cfg=CFG, trajectory_id=data["trajectory_id"],
        beam_changes=beam_changes_global,
    )
    _client.tx_codebook = mm_ds.tx_codebook
    _client.rx_codebook = mm_ds.rx_codebook
    return _client.to_client()

mm_strategy = AttackingHierarchicalFedAvg(
    cluster_map           = cluster_map_mm,
    traj_to_idx           = traj_to_idx_mm,
    attack_cluster        = MM_ATTACK_CLUSTER,
    attack_start_round    = MM_ATTACK_START_ROUND,
    eval_every            = 5,
    fraction_fit          = CFG["client_frac"],
    fraction_evaluate     = CFG["client_frac"],
    min_fit_clients       = max(2, int(len(mm_client_data) * CFG["client_frac"])),
    min_evaluate_clients  = max(2, int(len(mm_client_data) * CFG["client_frac"])),
    min_available_clients = len(mm_client_data),
    fit_metrics_aggregation_fn      = agg_metrics,
    evaluate_metrics_aggregation_fn = agg_metrics,
)

print(f"\n{'='*60}")
print(f"Run 3: Multimodal Hierarchical FL + Byzantine Attack")
print(f"  Rounds         : {MM_ROUNDS}")
print(f"  Clients        : {len(mm_client_data)}")
print(f"  Clusters       : {len(cluster_map_mm)}")
print(f"  Attack cluster : {MM_ATTACK_CLUSTER} (stale-weight injection)")
print(f"  Attack start   : round {MM_ATTACK_START_ROUND}")
print(f"  DDIL           : connectivity={CFG.get('ddil_connectivity_prob',0.9)*100:.0f}%  "
      f"doppler_std={CFG['ddil_doppler_noise_std']}  "
      f"pkt_drop={CFG['ddil_packet_dropout']*100:.0f}%  "
      f"blockage={CFG['ddil_signal_blockage_prob']*100:.0f}%")
print(f"  S2 thresh      : CH_member_sim < {AttackingHierarchicalFedAvg.CH_MEMBER_SIM_THRESHOLD}")
print(f"  FedProx μ      : {CFG['fedprox_mu']}")
print(f"  Multi-Krum f   : {AttackingHierarchicalFedAvg.MULTI_KRUM_F}")
print(f"{'='*60}\n")

mm_history = fl.simulation.start_simulation(
    client_fn        = mm_client_fn,
    num_clients      = len(mm_client_data),
    config           = fl.server.ServerConfig(num_rounds=MM_ROUNDS),
    strategy         = mm_strategy,
    client_resources = {"num_cpus": 2, "num_gpus": 0.0},
)
print("Run 3 complete.")

ATTACK_ACC       = float("nan")
ATTACK_TOP3      = float("nan")
ATTACK_GOPT_MAE  = float("nan")
if mm_history.metrics_distributed:
    _v = mm_history.metrics_distributed.get("beam_accuracy", [])
    if _v: ATTACK_ACC = max(v for _, v in _v)
    _v = mm_history.metrics_distributed.get("beam_top3_accuracy", [])
    if _v: ATTACK_TOP3 = max(v for _, v in _v)
    _v = mm_history.metrics_distributed.get("gopt_mae", [])
    if _v: ATTACK_GOPT_MAE = min(v for _, v in _v)
    print(f"  Best beam_accuracy  : {ATTACK_ACC:.4f}")
    print(f"  Best beam_top3_acc  : {ATTACK_TOP3:.4f}")
    print(f"  Best gopt_MAE       : {ATTACK_GOPT_MAE:.4f}")


In [ ]:
# ── Baseline 1: Random Beam (Lower Bound) ────────────────────────────────────
# A model that predicts uniformly at random. Sets the floor all learned models
# must beat. Expected top-1 = 1/Q_tx, top-3 = 3/Q_tx (for balanced beam dist).

Q_tx = CFG["Q_tx"]
np.random.seed(42)

random_top1, random_top3 = [], []
for client_obj in clients:
    y_beam = np.array([ds[i][1]["beam_index"] for i in client_obj.test_indices],
                      dtype=np.int32)
    n = len(y_beam)
    # top-1
    preds = np.random.randint(0, Q_tx, size=n)
    random_top1.append(float(np.mean(preds == y_beam)))
    # top-3: draw 3 distinct beams per sample
    top3_hits = [y_beam[i] in np.random.choice(Q_tx, 3, replace=False) for i in range(n)]
    random_top3.append(float(np.mean(top3_hits)))

RANDOM_ACC  = float(np.mean(random_top1))
RANDOM_TOP3 = float(np.mean(random_top3))

print(f"Random Beam Baseline (lower bound)")
print(f"  Theoretical top-1  : {1/Q_tx:.4f}  ({100/Q_tx:.1f}%)")
print(f"  Empirical  top-1   : {RANDOM_ACC:.4f}  ({RANDOM_ACC*100:.1f}%)")
print(f"  Empirical  top-3   : {RANDOM_TOP3:.4f}  ({RANDOM_TOP3*100:.1f}%)")


In [ ]:
# ── Run 1: Multimodal Flat FedAvg (No Hierarchy, No Attack, No DDIL) ─────────

_cfg_run1 = {**CFG,
    "ddil_packet_dropout": 0.0, "ddil_doppler_noise_std": 0.0,
    "ddil_signal_blockage_prob": 0.0, "ddil_connectivity_prob": 1.0,
}

if ray.is_initialized():
    ray.shutdown()

_r1_seed = 42
nr      = mm_channel_ds.nr
nt      = mm_channel_ds.nt
n_beams = CFG["Q_tx"]

def flat_mm_client_fn(context: fl.common.Context) -> fl.client.Client:
    _cidx = int(context.node_id) % len(mm_client_data)
    data  = mm_client_data[_cidx]
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.random.set_seed(_r1_seed + _cidx)
    np.random.seed(_r1_seed + _cidx)
    model = MultimodalBeamModel(nr=nr, nt=nt, n_beams=n_beams, dropout=0.1)
    _client = MultimodalBeamFlowerClient(
        model=model, dataset=mm_ds,
        train_indices=data["train_indices"], test_indices=data["test_indices"],
        cfg=_cfg_run1, trajectory_id=data["trajectory_id"],
        beam_changes=beam_changes_global,
    )
    _client.tx_codebook = mm_ds.tx_codebook
    _client.rx_codebook = mm_ds.rx_codebook
    return _client.to_client()

flat_strategy = fl.server.strategy.FedAvg(
    fraction_fit          = _cfg_run1["client_frac"],
    fraction_evaluate     = _cfg_run1["client_frac"],
    min_fit_clients       = max(2, int(len(mm_client_data) * _cfg_run1["client_frac"])),
    min_evaluate_clients  = max(2, int(len(mm_client_data) * _cfg_run1["client_frac"])),
    min_available_clients = len(mm_client_data),
    fit_metrics_aggregation_fn      = agg_metrics,
    evaluate_metrics_aggregation_fn = agg_metrics,
)

FLAT_ROUNDS = 100
print(f"\n{'='*60}")
print(f"Run 1: Multimodal Flat FedAvg (No Hierarchy, No Attack, No DDIL)")
print(f"  Rounds  : {FLAT_ROUNDS}  |  Clients: {len(mm_client_data)}")
print(f"{'='*60}\n")

flat_mm_history = fl.simulation.start_simulation(
    client_fn        = flat_mm_client_fn,
    num_clients      = len(mm_client_data),
    config           = fl.server.ServerConfig(num_rounds=FLAT_ROUNDS),
    strategy         = flat_strategy,
    client_resources = {"num_cpus": 2, "num_gpus": 0.0},
)
print("Run 1 complete.")

FLAT_ACC = FLAT_TOP3 = FLAT_GOPT_MAE = float("nan")
if flat_mm_history.metrics_distributed:
    _v = flat_mm_history.metrics_distributed.get("beam_accuracy", [])
    if _v: FLAT_ACC = max(v for _, v in _v)
    _v = flat_mm_history.metrics_distributed.get("beam_top3_accuracy", [])
    if _v: FLAT_TOP3 = max(v for _, v in _v)
    _v = flat_mm_history.metrics_distributed.get("gopt_mae", [])
    if _v: FLAT_GOPT_MAE = min(v for _, v in _v)
    print(f"  beam_accuracy : {FLAT_ACC:.4f}  top3: {FLAT_TOP3:.4f}")
    print(f"  gopt_MAE (dB): {FLAT_GOPT_MAE:.4f}")

In [ ]:
# ── Run 2: Multimodal Hierarchical FL, No Attack, With DDIL ──────────────────
# 2-level hierarchical aggregation using multimodal model (CSI + LiDAR + IMU).
# DDIL channel impairments enabled. No Byzantine attack.
# Shows hierarchical FL benefit under realistic UAV channel conditions.

if ray.is_initialized():
    ray.shutdown()
    print("Ray shutdown")

_nr2      = mm_channel_ds.nr
_nt2      = mm_channel_ds.nt
_n_beams2 = CFG["Q_tx"]

HIER_ROUNDS     = 100
HIER_EVAL_EVERY = 5

_hier_cluster_map = build_clusters(mm_clients_raw)
_hier_traj_to_idx = {mm_clients_raw[k].trajectory_id: k
                     for k in range(len(mm_clients_raw))}

def hier_client_fn(context: fl.common.Context) -> fl.client.Client:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    client_idx = int(context.node_id) % len(mm_client_data)
    data = mm_client_data[client_idx]
    model = MultimodalBeamModel(nr=_nr2, nt=_nt2, n_beams=_n_beams2, dropout=0.1)
    _client = MultimodalBeamFlowerClient(
        model=model, dataset=mm_ds,
        train_indices=data["train_indices"], test_indices=data["test_indices"],
        cfg=CFG, trajectory_id=data["trajectory_id"],
        beam_changes=beam_changes_global,
    )
    _client.tx_codebook = mm_ds.tx_codebook
    _client.rx_codebook = mm_ds.rx_codebook
    return _client.to_client()

hier_strategy = HierarchicalFedAvg(
    cluster_map                     = _hier_cluster_map,
    traj_to_idx                     = _hier_traj_to_idx,
    eval_every                      = HIER_EVAL_EVERY,
    ddil_connectivity_prob          = CFG.get("ddil_connectivity_prob", 0.90),
    fraction_fit                    = CFG["client_frac"],
    fraction_evaluate               = CFG["client_frac"],
    min_fit_clients                 = max(2, int(len(mm_client_data) * CFG["client_frac"])),
    min_evaluate_clients            = max(2, int(len(mm_client_data) * CFG["client_frac"])),
    min_available_clients           = len(mm_client_data),
    fit_metrics_aggregation_fn      = agg_metrics,
    evaluate_metrics_aggregation_fn = agg_metrics,
)

print(f"\n{'='*60}")
print(f"Run 2: Multimodal Hierarchical FL + DDIL (No Attack)")
print(f"  Rounds    : {HIER_ROUNDS}")
print(f"  Clients   : {len(mm_client_data)}")
print(f"  Clusters  : {len(_hier_cluster_map)}")
print(f"  Attack    : None")
print(f"  DDIL      : connectivity={CFG.get('ddil_connectivity_prob',0.9)*100:.0f}%  "
      f"doppler_std={CFG['ddil_doppler_noise_std']}  "
      f"pkt_drop={CFG['ddil_packet_dropout']*100:.0f}%  "
      f"blockage={CFG['ddil_signal_blockage_prob']*100:.0f}%")
print(f"  FedProx μ : {CFG['fedprox_mu']}")
print(f"  Multi-Krum: f={HierarchicalFedAvg.MULTI_KRUM_F}")
print(f"  S2 thresh : CH_member_sim < {HierarchicalFedAvg.CH_MEMBER_SIM_THRESHOLD}")
print(f"{'='*60}\n")

hier_history = fl.simulation.start_simulation(
    client_fn        = hier_client_fn,
    num_clients      = len(mm_client_data),
    config           = fl.server.ServerConfig(num_rounds=HIER_ROUNDS),
    strategy         = hier_strategy,
    client_resources = {"num_cpus": 2, "num_gpus": 0.0},
)
print("Run 2 complete.")

HIER_ACC       = float("nan")
HIER_TOP3      = float("nan")
HIER_GOPT_MAE  = float("nan")
if hier_history.metrics_distributed:
    _v = hier_history.metrics_distributed.get("beam_accuracy", [])
    if _v: HIER_ACC = max(v for _, v in _v)
    _v = hier_history.metrics_distributed.get("beam_top3_accuracy", [])
    if _v: HIER_TOP3 = max(v for _, v in _v)
    _v = hier_history.metrics_distributed.get("gopt_mae", [])
    if _v: HIER_GOPT_MAE = min(v for _, v in _v)
    print(f"  Best beam_accuracy  : {HIER_ACC:.4f}")
    print(f"  Best beam_top3_acc  : {HIER_TOP3:.4f}")
    print(f"  Best gopt_MAE       : {HIER_GOPT_MAE:.4f}")


In [ ]:
# ── Run 2b: Multimodal Hierarchical FL, No Attack, No DDIL ──────────────────
# Same as Run 2 but with DDIL fully disabled (connectivity_prob=1.0, noise=0).
# Purpose: isolates the hierarchy benefit from the DDIL penalty.
# Comparison: Run2b vs Run1 (flat, no DDIL) = pure hierarchy gain.
#             Run2  vs Run2b                 = DDIL cost under hierarchy.

if ray.is_initialized():
    ray.shutdown()
    print("Ray shutdown")

_nr2b      = mm_channel_ds.nr
_nt2b      = mm_channel_ds.nt
_n_beams2b = CFG["Q_tx"]

HIER2B_ROUNDS     = 100
HIER2B_EVAL_EVERY = 5

_hier2b_cluster_map = build_clusters(mm_clients_raw)
_hier2b_traj_to_idx = {mm_clients_raw[k].trajectory_id: k
                       for k in range(len(mm_clients_raw))}

# DDIL-disabled config: full connectivity, zero noise
_cfg_no_ddil = {
    **CFG,
    "ddil_connectivity_prob":   1.0,   # no dropout
    "ddil_doppler_noise_std":   0.0,   # no Doppler
    "ddil_packet_dropout":      0.0,   # no packet loss
    "ddil_signal_blockage_prob":0.0,   # no blockage
}

def hier2b_client_fn(context: fl.common.Context) -> fl.client.Client:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    client_idx = int(context.node_id) % len(mm_client_data)
    data = mm_client_data[client_idx]
    model = MultimodalBeamModel(nr=_nr2b, nt=_nt2b, n_beams=_n_beams2b, dropout=0.1)
    _client = MultimodalBeamFlowerClient(
        model=model, dataset=mm_ds,
        train_indices=data["train_indices"], test_indices=data["test_indices"],
        cfg=_cfg_no_ddil, trajectory_id=data["trajectory_id"],
        beam_changes=beam_changes_global,
    )
    _client.tx_codebook = mm_ds.tx_codebook
    _client.rx_codebook = mm_ds.rx_codebook
    return _client.to_client()

hier2b_strategy = HierarchicalFedAvg(
    cluster_map                     = _hier2b_cluster_map,
    traj_to_idx                     = _hier2b_traj_to_idx,
    eval_every                      = HIER2B_EVAL_EVERY,
    ddil_connectivity_prob          = 1.0,   # DDIL disabled
    fraction_fit                    = CFG["client_frac"],
    fraction_evaluate               = CFG["client_frac"],
    min_fit_clients                 = max(2, int(len(mm_client_data) * CFG["client_frac"])),
    min_evaluate_clients            = max(2, int(len(mm_client_data) * CFG["client_frac"])),
    min_available_clients           = len(mm_client_data),
    fit_metrics_aggregation_fn      = agg_metrics,
    evaluate_metrics_aggregation_fn = agg_metrics,
)

print(f"\n{'='*60}")
print(f"Run 2b: Multimodal Hierarchical FL — DDIL DISABLED (clean channel)")
print(f"  Rounds    : {HIER2B_ROUNDS}")
print(f"  Clients   : {len(mm_client_data)}")
print(f"  Clusters  : {len(_hier2b_cluster_map)}")
print(f"  Attack    : None")
print(f"  DDIL      : DISABLED (connectivity=100%, noise=0)")
print(f"  Purpose   : isolate hierarchy gain vs Run 1 (flat FedAvg, same conditions)")
print(f"{'='*60}\n")

hier2b_history = fl.simulation.start_simulation(
    client_fn        = hier2b_client_fn,
    num_clients      = len(mm_client_data),
    config           = fl.server.ServerConfig(num_rounds=HIER2B_ROUNDS),
    strategy         = hier2b_strategy,
    client_resources = {"num_cpus": 2, "num_gpus": 0.0},
)
print("Run 2b complete.")

HIER2B_ACC       = float("nan")
HIER2B_TOP3      = float("nan")
HIER2B_GOPT_MAE  = float("nan")
if hier2b_history.metrics_distributed:
    _v = hier2b_history.metrics_distributed.get("beam_accuracy", [])
    if _v: HIER2B_ACC = max(v for _, v in _v)
    _v = hier2b_history.metrics_distributed.get("beam_top3_accuracy", [])
    if _v: HIER2B_TOP3 = max(v for _, v in _v)
    _v = hier2b_history.metrics_distributed.get("gopt_mae", [])
    if _v: HIER2B_GOPT_MAE = min(v for _, v in _v)
    print(f"  Best beam_accuracy  : {HIER2B_ACC:.4f}")
    print(f"  Best beam_top3_acc  : {HIER2B_TOP3:.4f}")
    print(f"  Best gopt_MAE       : {HIER2B_GOPT_MAE:.4f}")


In [ ]:
# ── Baseline 3: Centralized SGD (Upper Bound / Genie) ────────────────────────
# Pools ALL multimodal client data and trains a single model locally.
# Establishes the accuracy ceiling: what FL gives up vs. full data access.
# Uses identical model architecture, loss weights, and LR as FL clients.

tf.keras.backend.clear_session()
_cent_nr     = mm_channel_ds.nr
_cent_nt     = mm_channel_ds.nt
_cent_nbeams = CFG["Q_tx"]

_cent_model = MultimodalBeamModel(nr=_cent_nr, nt=_cent_nt,
                                  n_beams=_cent_nbeams, dropout=0.1)
_cent_opt   = keras.optimizers.Adam(CFG["lr"])

# Pool all sample indices from all multimodal clients
_all_tr_idx = [i for c in mm_client_data for i in c["train_indices"]]
_all_te_idx = [i for c in mm_client_data for i in c["test_indices"]]
print(f"Centralized SGD: {len(_all_tr_idx)} train, {len(_all_te_idx)} test samples")


def _cent_load(indices, split="train", _gopt_stats_ref=[None]):
    """Load pooled multimodal data using the same pipeline as MultimodalBeamFlowerClient."""
    global _MM_ACTOR_CACHE
    missing = [i for i in indices if i not in _MM_ACTOR_CACHE]
    for i in missing:
        csi, lidar, imu, labels = mm_ds[i]
        _MM_ACTOR_CACHE[i] = (csi, lidar, imu, labels)

    CSI, LIDAR, IMU, y_beam, y_gopt_raw, y_change = [], [], [], [], [], []
    for i in indices:
        csi, lidar, imu, labels = _MM_ACTOR_CACHE[i]
        CSI.append(csi); LIDAR.append(lidar); IMU.append(imu)
        y_beam.append(int(labels["beam_index"]))
        y_gopt_raw.append(float(labels.get("g_opt", 1.0)))
        y_change.append(int(beam_changes_global.get(i, 0)))

    csi_arr   = np.stack(CSI).astype(np.float32)
    lidar_arr = np.stack(LIDAR).astype(np.float32)
    imu_arr   = np.stack(IMU).astype(np.float32)

    # CSI per-sample normalisation
    csi_arr = ((csi_arr - csi_arr.mean(axis=(1,2), keepdims=True)) /
               (csi_arr.std(axis=(1,2), keepdims=True) + 1e-8))

    # gopt: dB conversion + z-normalise (same as MultimodalBeamFlowerClient)
    gopt_db = 10.0 * np.log10(np.clip(np.array(y_gopt_raw, np.float32), 1e-12, None))
    if split == "train":
        _gopt_stats_ref[0] = (float(gopt_db.mean()), float(gopt_db.std()) + 1e-8)
    mu_g, sigma_g = (_gopt_stats_ref[0] if _gopt_stats_ref[0]
                     else (float(gopt_db.mean()), float(gopt_db.std()) + 1e-8))
    gopt_arr = ((gopt_db - mu_g) / sigma_g).astype(np.float32)

    return (csi_arr, lidar_arr, imu_arr,
            np.array(y_beam,   np.int32),
            gopt_arr,
            np.array(y_change, np.int32))


csi_tr, lid_tr, imu_tr, yb_tr, yg_tr, yc_tr = _cent_load(_all_tr_idx, split="train")
csi_te, lid_te, imu_te, yb_te, yg_te, yc_te = _cent_load(_all_te_idx, split="test")
# Capture sigma for dB denormalization of gopt MAE
_cent_sigma_g = (_cent_load.__defaults__[-1][0][1]
                 if _cent_load.__defaults__[-1][0] is not None else 1.0)

# Build model
_cent_model(tf.constant(csi_tr[:1], tf.float32),
            tf.constant(lid_tr[:1], tf.float32),
            tf.constant(imu_tr[:1], tf.float32), training=False)

w       = MultimodalBeamFlowerClient.LOSS_WEIGHTS
_bs     = 128
_smooth = CFG.get("label_smoothing", 0.0)
_pw_chg = CFG.get("pos_weight_change", 1.0)
CENT_EPOCHS = 50
cent_log    = []

print(f"Training {CENT_EPOCHS} epochs, batch_size={_bs}")
for _ep in range(CENT_EPOCHS):
    _perm   = np.random.permutation(len(csi_tr))
    _losses = []
    for _s in range(0, len(csi_tr), _bs):
        _e   = min(_s + _bs, len(csi_tr))
        _idx = _perm[_s:_e]
        _cb  = tf.constant(csi_tr[_idx],                   tf.float32)
        _lb  = tf.constant(lid_tr[_idx],                   tf.float32)
        _ib  = tf.constant(imu_tr[_idx],                   tf.float32)
        _bb  = tf.constant(yb_tr[_idx],                    tf.int32)
        _gb  = tf.constant(yg_tr[_idx],                    tf.float32)
        _chb = tf.constant(yc_tr[_idx].astype(np.float32), tf.float32)
        with tf.GradientTape() as tape:
            _outs = _cent_model(_cb, _lb, _ib, training=True)
            _nc   = _outs["beam"].shape[-1]
            if _smooth > 0:
                _oh  = tf.one_hot(_bb, _nc)
                _sl  = _oh * (1 - _smooth) + _smooth / tf.cast(_nc, tf.float32)
                _l_b = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(
                    labels=_sl, logits=_outs["beam"]))
            else:
                _l_b = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(
                    labels=_bb, logits=_outs["beam"]))
            _l_g  = tf.reduce_mean(tf.square(_outs["gopt"] - _gb))
            _l_ch = tf.reduce_mean(tf.nn.weighted_cross_entropy_with_logits(
                labels=_chb, logits=_outs["beam_change"], pos_weight=_pw_chg))
            _loss = w["beam"] * _l_b + w["gopt"] * _l_g + w["change"] * _l_ch
        _grads = tape.gradient(_loss, _cent_model.trainable_weights)
        _grads = [g if g is not None else tf.zeros_like(v)
                  for g, v in zip(_grads, _cent_model.trainable_weights)]
        _cent_opt.apply_gradients(zip(_grads, _cent_model.trainable_weights))
        _losses.append(float(_loss.numpy()))

    if (_ep + 1) % 5 == 0:
        _all_logits, _all_gopt = [], []
        for _ss in range(0, len(csi_te), _bs):
            _se = min(_ss + _bs, len(csi_te))
            _o  = _cent_model(
                tf.constant(csi_te[_ss:_se], tf.float32),
                tf.constant(lid_te[_ss:_se], tf.float32),
                tf.constant(imu_te[_ss:_se], tf.float32), training=False)
            _all_logits.append(_o["beam"].numpy())
            _all_gopt.append(_o["gopt"].numpy())
        _logits = np.concatenate(_all_logits)
        _gp     = np.concatenate(_all_gopt)
        _preds  = np.argmax(_logits, axis=1)
        _acc    = float(np.mean(_preds == yb_te))
        _top3   = float(np.mean([yb_te[i] in np.argsort(_logits[i])[-3:]
                                 for i in range(len(yb_te))]))
        _gerr   = _gp - yg_te
        _gmae   = float(np.mean(np.abs(_gerr))) * _cent_sigma_g   # dB
        cent_log.append((_ep+1, _acc, _top3, _gmae, float(np.mean(_losses))))
        print(f"  Epoch {_ep+1:3d}: loss={np.mean(_losses):.3f}  "
              f"top-1={_acc:.4f}  top-3={_top3:.4f}  "
              f"gopt_MAE={_gmae:.4f} dB")

CENT_ACC       = max(h[1] for h in cent_log) if cent_log else float("nan")
CENT_TOP3      = max(h[2] for h in cent_log) if cent_log else float("nan")
CENT_GOPT_MAE  = min(h[3] for h in cent_log) if cent_log else float("nan")
print(f"\nCentralized SGD (upper bound):")
print(f"  top-1={CENT_ACC:.4f}  top-3={CENT_TOP3:.4f}  "
      f"gopt_MAE={CENT_GOPT_MAE:.4f} dB")


In [ ]:
# ── Results Comparison Table ──────────────────────────────────────────────────
# Side-by-side summary: lower bound → Run 1 → Run 2 → Run 3 → upper bound

import numpy as np
import pandas as pd


def _best_from_history(history, key, mode="max"):
    vals = []
    for _, v in getattr(history, "metrics_distributed", {}).get(key, []):
        vals.append(v)
    for _, v in getattr(history, "metrics_centralized", {}).get(key, []):
        vals.append(v)
    if not vals:
        return float("nan")
    return float(np.max(vals)) if mode == "max" else float(np.min(vals))


def _fmt(v, pct=True, decimals=4):
    if np.isnan(v): return "N/A"
    return f"{v*100:.2f}%" if pct else f"{v:.{decimals}f}"


# ── Gather metrics from each run ─────────────────────────────────────────────
# Run 1: Flat FedAvg (FLAT_ACC, FLAT_TOP3, FLAT_GOPT_MAE)
# Run 2: Hierarchical FL no attack (HIER_ACC, HIER_TOP3, HIER_GOPT_MAE)
# Run 3: Hierarchical FL + attack (ATTACK_ACC, ATTACK_TOP3, ATTACK_GOPT_MAE)

# Safe fallback if a run hasn't been executed yet
_r1_acc    = FLAT_ACC    if "FLAT_ACC"    in dir() else float("nan")
_r1_top3   = FLAT_TOP3   if "FLAT_TOP3"  in dir() else float("nan")
_r1_gmae   = FLAT_GOPT_MAE   if "FLAT_GOPT_MAE"   in dir() else float("nan")

_r2_acc    = HIER_ACC    if "HIER_ACC"    in dir() else float("nan")
_r2_top3   = HIER_TOP3   if "HIER_TOP3"  in dir() else float("nan")
_r2_gmae   = HIER_GOPT_MAE   if "HIER_GOPT_MAE"   in dir() else float("nan")

_r3_acc    = ATTACK_ACC    if "ATTACK_ACC"    in dir() else float("nan")
_r3_top3   = ATTACK_TOP3   if "ATTACK_TOP3"  in dir() else float("nan")
_r3_gmae   = ATTACK_GOPT_MAE  if "ATTACK_GOPT_MAE"  in dir() else float("nan")

# ── Build comparison dataframe ────────────────────────────────────────────────
rows = [
    {
        "Method":        "Random Beam (lower bound)",
        "Top-1 Acc":     _fmt(RANDOM_ACC)  if "RANDOM_ACC"  in dir() else "N/A",
        "Top-3 Acc":     _fmt(RANDOM_TOP3) if "RANDOM_TOP3" in dir() else "N/A",
        "gopt MAE (dB)": "N/A",
        "Notes":         "Empirical frequency baseline; no learning",
    },
    {
        "Method":        "Run 1: Flat FedAvg (no DDIL)",
        "Top-1 Acc":     _fmt(_r1_acc),
        "Top-3 Acc":     _fmt(_r1_top3),
        "gopt MAE (dB)": _fmt(_r1_gmae, pct=False),
        "Notes":         "Standard FedAvg, 30 rounds, clean channel",
    },
    {
        "Method":        "Run 2: Hierarchical FL + DDIL",
        "Top-1 Acc":     _fmt(_r2_acc),
        "Top-3 Acc":     _fmt(_r2_top3),
        "gopt MAE (dB)": _fmt(_r2_gmae, pct=False),
        "Notes":         "2-level hierarchy, DDIL enabled, no attack",
    },
    {
        "Method":        "Run 3: Hierarchical FL + DDIL + Byzantine (ours)",
        "Top-1 Acc":     _fmt(_r3_acc),
        "Top-3 Acc":     _fmt(_r3_top3),
        "gopt MAE (dB)": _fmt(_r3_gmae, pct=False),
        "Notes":         "2-level hierarchy, S2∧S3 detector, CH0 stale-injection @R11",
    },
    {
        "Method":        "Centralized SGD (upper bound)",
        "Top-1 Acc":     _fmt(CENT_ACC)  if "CENT_ACC"  in dir() else "N/A",
        "Top-3 Acc":     _fmt(CENT_TOP3) if "CENT_TOP3" in dir() else "N/A",
        "gopt MAE (dB)": "N/A",
        "Notes":         "All data pooled; genie bound — not achievable in FL",
    },
]

df = pd.DataFrame(rows, columns=["Method", "Top-1 Acc", "Top-3 Acc", "gopt MAE (dB)", "Notes"])
df.index = df.index + 1

print("=" * 100)
print("  BEAM PREDICTION — METHOD COMPARISON SUMMARY")
print("=" * 100)
print(df.to_string(index=True))
print("=" * 100)

# ── Gap analysis ──────────────────────────────────────────────────────────────
if not np.isnan(_r3_acc):
    _rand = RANDOM_ACC if "RANDOM_ACC" in dir() else float("nan")
    _cent = CENT_ACC   if "CENT_ACC"   in dir() else float("nan")
    print(f"\nGap analysis (Run 3 = full system):")
    if not np.isnan(_r1_acc):
        print(f"  vs Flat FedAvg   : {(_r3_acc - _r1_acc)*100:+.2f} pp top-1")
    if not np.isnan(_r2_acc):
        print(f"  vs Hier (no atk) : {(_r3_acc - _r2_acc)*100:+.2f} pp top-1  "
              f"(attack cost recovered by detector)")
    if not np.isnan(_rand):
        print(f"  vs Random        : {(_r3_acc - _rand)*100:+.2f} pp above lower bound")
    if not np.isnan(_cent):
        print(f"  vs Centralized   : {(_cent - _r3_acc)*100:+.2f} pp gap to genie bound")